[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/linear_algebra/07_canonical_forms_and_svd/exercises.ipynb)

# Module 07 — Exercises: Canonical Forms and the Singular Value Decomposition

Forty-two solved problems in four tiers. Every problem carries a statement, a one-line intuition, a
stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is numeric or
algorithmic — a code cell that recomputes it.

Theorem and proof numbers refer to [first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): singular values descending, transposes written
$A^{\top}$, norms written $\lVert \cdot \rVert$.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps
print(f"machine epsilon = {EPS:.4e}")

machine epsilon = 2.2204e-16


## L0 — Concept Checks

### Problem L0.1 — Singular values of a rank-one matrix

**Statement.** Does $A = \begin{pmatrix} 1 & 1 \\ 0 & 0 \end{pmatrix}$ have an SVD? Give its
singular values.

**Intuition.** Theorem 4.3 has no hypotheses, so the answer to the first question is yes for every
matrix; the singular values are the square roots of the eigenvalues of $A^{\top}A$.

**Solution.**

*Step 1.* $A^{\top}A = \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}$.

*Step 2.* $\det(A^{\top}A - \lambda I) = (1-\lambda)^2 - 1 = \lambda(\lambda - 2)$, so the
eigenvalues are $2$ and $0$.

$$
\boxed{\sigma_1 = \sqrt{2}, \qquad \sigma_2 = 0}
$$

**Key takeaway.** Existence of the SVD needs nothing: rectangular, singular and non-normal matrices
all have one.

In [2]:
A = np.array([[1.0, 1.0], [0.0, 0.0]])
print("A^T A       :\n", A.T @ A)
print("singular values:", np.linalg.svd(A, compute_uv=False), "  sqrt2 =", np.sqrt(2))
assert np.allclose(np.linalg.svd(A, compute_uv=False), [np.sqrt(2), 0.0])

A^T A       :
 [[1. 1.]
 [1. 1.]]
singular values: [1.4142 0.    ]   sqrt2 = 1.4142135623730951


### Problem L0.2 — Pseudoinverse of a $1 \times 1$ matrix

**Statement.** Let $A = [x]$ with $x \in \mathbb{R}$. Find $A^{+}$.

**Intuition.** The pseudoinverse inverts what it can and sends the rest to zero.

**Solution.**

*Step 1.* If $x \neq 0$ the matrix is invertible, and $[1/x]$ satisfies all four Penrose
conditions trivially.

*Step 2.* If $x = 0$ then $\Sigma = [0]$, so $\Sigma^{+} = [0]$ by Definition 3.6, and
$A^{+} = [0]$. Proof 5.8 shows any other value breaks (P2).

$$
\boxed{A^{+} = [1/x] \text{ if } x \neq 0, \qquad A^{+} = [0] \text{ if } x = 0}
$$

**Key takeaway.** $A \mapsto A^{+}$ is not continuous at a rank drop: $1/x$ blows up as
$x \to 0$, yet $0^{+} = 0$.

In [3]:
for x in (2.0, -0.5, 0.0):
    print(f"x = {x:5.2f}   pinv = {np.linalg.pinv(np.array([[x]]))[0, 0]:8.4f}")
assert np.linalg.pinv(np.array([[0.0]]))[0, 0] == 0.0
assert np.isclose(np.linalg.pinv(np.array([[2.0]]))[0, 0], 0.5)

x =  2.00   pinv =   0.5000
x = -0.50   pinv =  -2.0000
x =  0.00   pinv =   0.0000


### Problem L0.3 — Singular values of an outer product of unit vectors

**Statement.** Let $u \in \mathbb{R}^m$ and $v \in \mathbb{R}^n$ be unit vectors and
$A = uv^{\top}$. Give the singular values of $A$.

**Intuition.** $uv^{\top}$ is already written in the form $\sigma_1 u_1 v_1^{\top}$ with
$\sigma_1 = 1$.

**Solution.**

*Step 1.* $A^{\top}A = v(u^{\top}u)v^{\top} = vv^{\top}$, the orthogonal projector onto
$\operatorname{span}(v)$.

*Step 2.* A projector of rank $1$ has eigenvalues $1$ and $0$, so $\sigma_1 = 1$ and all other
singular values vanish.

$$
\boxed{\sigma_1 = 1, \qquad \sigma_i = 0 \ \text{ for } i \ge 2}
$$

**Key takeaway.** For a general outer product $A = ab^{\top}$ the single singular value is
$\lVert a \rVert \lVert b \rVert$.

In [4]:
u = rng.standard_normal(5)
v = rng.standard_normal(3)
u, v = u / np.linalg.norm(u), v / np.linalg.norm(v)
print("singular values of u v^T:", np.linalg.svd(np.outer(u, v), compute_uv=False))
a, b = 3.0 * u, 2.0 * v
print("singular values of (3u)(2v)^T:", np.linalg.svd(np.outer(a, b), compute_uv=False), " expect 6")
assert np.allclose(np.linalg.svd(np.outer(u, v), compute_uv=False), [1.0, 0.0, 0.0])
assert np.isclose(np.linalg.svd(np.outer(a, b), compute_uv=False)[0], 6.0)

singular values of u v^T: [1. 0. 0.]
singular values of (3u)(2v)^T: [6. 0. 0.]  expect 6


### Problem L0.4 — Condition number of a diagonal matrix

**Statement.** Compute $\kappa_2(A)$ for $A = \operatorname{diag}(3, 10^{-4})$.

**Intuition.** A diagonal matrix is already in SVD form up to signs, so its singular values are the
absolute diagonal entries.

**Solution.**

*Step 1.* $\sigma_1 = 3$, $\sigma_2 = 10^{-4}$.

*Step 2.* $\kappa_2(A) = \sigma_1/\sigma_2$ by Definition 3.5.

$$
\boxed{\kappa_2(A) = 3 \times 10^{4} = 30000}
$$

**Key takeaway.** The condition number measures eccentricity of the image ellipsoid, not size: it
is invariant under $A \mapsto cA$.

In [5]:
A = np.diag([3.0, 1e-4])
print("singular values:", np.linalg.svd(A, compute_uv=False))
print("kappa_2        :", np.linalg.cond(A))
print("kappa_2 of 100A:", np.linalg.cond(100 * A), " (scale invariant)")
assert np.isclose(np.linalg.cond(A), 3e4)
assert np.isclose(np.linalg.cond(100 * A), np.linalg.cond(A))

singular values: [3.     0.0001]
kappa_2        : 30000.0
kappa_2 of 100A: 30000.0  (scale invariant)


### Problem L0.5 — Best rank-one approximation of a diagonal matrix

**Statement.** Give the best rank-one approximation of $A = \operatorname{diag}(2, 1)$ and its
error in both norms.

**Intuition.** Theorem 4.6 says: keep the largest singular value, drop the rest.

**Solution.**

*Step 1.* $\sigma_1 = 2$ with $u_1 = v_1 = e_1$; $\sigma_2 = 1$.

*Step 2.* $A_1 = \sigma_1 u_1 v_1^{\top} = \operatorname{diag}(2, 0)$.

*Step 3.* $A - A_1 = \operatorname{diag}(0,1)$, whose only singular value is $1 = \sigma_2$.

$$
\boxed{A_1 = \begin{pmatrix} 2 & 0 \\ 0 & 0\end{pmatrix}, \qquad \lVert A - A_1 \rVert_{\mathrm{op}} = \lVert A - A_1 \rVert_F = 1}
$$

**Key takeaway.** With one discarded singular value the two norms of the error coincide; they
differ only when more than one term is dropped.

In [6]:
A = np.diag([2.0, 1.0])
U, s, Vt = np.linalg.svd(A)
A1 = s[0] * np.outer(U[:, 0], Vt[0])
print("A_1 =\n", A1)
print("error op, F:", np.linalg.norm(A - A1, 2), np.linalg.norm(A - A1), "  sigma_2 =", s[1])
assert np.allclose(A1, np.diag([2.0, 0.0]))
assert np.isclose(np.linalg.norm(A - A1, 2), 1.0) and np.isclose(np.linalg.norm(A - A1), 1.0)

A_1 =
 [[2. 0.]
 [0. 0.]]
error op, F: 1.0 1.0   sigma_2 = 1.0


### Problem L0.6 — SVD of the transpose

**Statement.** Given $A = U\Sigma V^{\top}$, write down an SVD of $A^{\top}$ and compare the
singular values.

**Intuition.** Transposing swaps the roles of domain and codomain, hence of $U$ and $V$.

**Solution.**

*Step 1.* $A^{\top} = (U\Sigma V^{\top})^{\top} = V \Sigma^{\top} U^{\top}$.

*Step 2.* $V$ and $U$ are orthogonal and $\Sigma^{\top}$ is diagonal with the same non-negative
entries in the same order, so this is a valid SVD.

$$
\boxed{A^{\top} = V\Sigma^{\top}U^{\top}, \qquad \sigma_i(A^{\top}) = \sigma_i(A)}
$$

**Key takeaway.** $\lVert A \rVert_{\mathrm{op}} = \lVert A^{\top} \rVert_{\mathrm{op}}$ and
$\lVert A \rVert_F = \lVert A^{\top} \rVert_F$ are corollaries.

In [7]:
A = rng.standard_normal((5, 3))
print("sv(A)  :", np.linalg.svd(A, compute_uv=False))
print("sv(A^T):", np.linalg.svd(A.T, compute_uv=False))
assert np.allclose(np.linalg.svd(A, compute_uv=False), np.linalg.svd(A.T, compute_uv=False))
assert np.isclose(np.linalg.norm(A, 2), np.linalg.norm(A.T, 2))

sv(A)  : [3.1735 1.6102 1.1592]
sv(A^T): [3.1735 1.6102 1.1592]


### Problem L0.7 — Jordan form of a single Jordan block

**Statement.** Find the Jordan canonical form of
$A = \begin{pmatrix} 2 & 1 & 0 \\ 0 & 2 & 1 \\ 0 & 0 & 2 \end{pmatrix}$.

**Intuition.** The matrix is already a Jordan block, and Theorem 4.2 makes the form unique, so
there is nothing to do.

**Solution.**

*Step 1.* $p_A(z) = (z-2)^3$, so $\lambda = 2$ with $m_2 = 3$.

*Step 2.* $\operatorname{rank}(A - 2I) = 2$, so $g_2 = 1$: one block, of size $3$.

$$
\boxed{J = J_3(2) = \begin{pmatrix} 2 & 1 & 0 \\ 0 & 2 & 1 \\ 0 & 0 & 2 \end{pmatrix}}
$$

**Key takeaway.** By Theorem 4.2 the block count is $g_\lambda$; a matrix with a one-dimensional
eigenspace has a single block filling the whole multiplicity.

In [8]:
A = np.array([[2.0, 1, 0], [0, 2, 1], [0, 0, 2]])
ranks = [int(np.linalg.matrix_rank(np.linalg.matrix_power(A - 2 * np.eye(3), j))) for j in range(4)]
print("rank (A-2I)^j :", ranks)
print("blocks of size >= j:", [ranks[j - 1] - ranks[j] for j in range(1, 4)])
assert ranks == [3, 2, 1, 0]
assert [ranks[j - 1] - ranks[j] for j in range(1, 4)] == [1, 1, 1]

rank (A-2I)^j : [3, 2, 1, 0]
blocks of size >= j: [1, 1, 1]


### Problem L0.8 — Jordan form of a non-zero square-zero matrix

**Statement.** Let $A \in \mathbb{C}^{2 \times 2}$ satisfy $A^2 = 0$ and $A \neq 0$. Find $J$.

**Intuition.** $A$ is nilpotent, so its only eigenvalue is $0$; and $A \neq 0$ rules out the
diagonal form.

**Solution.**

*Step 1.* $A^2 = 0$ forces every eigenvalue to be $0$, so $p_A(z) = z^2$.

*Step 2.* $A \neq 0$ gives $\operatorname{rank}(A) = 1$, so $r_1 = 1$ and by Theorem 4.2 the
number of blocks is $r_0 - r_1 = 2 - 1 = 1$.

$$
\boxed{J = \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix}}
$$

**Key takeaway.** For a nilpotent matrix the Jordan form is determined by the rank sequence alone,
since the only eigenvalue is $0$.

In [9]:
A = np.array([[1.0, -1.0], [1.0, -1.0]])
print("A^2 =\n", A @ A)
print("rank A =", np.linalg.matrix_rank(A), "  eigenvalues:", np.linalg.eigvals(A))
print("blocks =", 2 - np.linalg.matrix_rank(A))
assert np.allclose(A @ A, 0.0) and np.linalg.matrix_rank(A) == 1
assert 2 - np.linalg.matrix_rank(A) == 1

A^2 =
 [[0. 0.]
 [0. 0.]]
rank A = 1   eigenvalues: [0.+0.j 0.-0.j]
blocks = 1


## L1 — Foundations

### Problem L1.1 — Frobenius norm from the singular values

**Statement.** Prove that $\lVert A \rVert_F^2 = \sum_{i=1}^{r} \sigma_i^2$ for every
$A \in \mathbb{R}^{m \times n}$ of rank $r$.

**Intuition.** The Frobenius norm is invariant under orthogonal factors, so it can be computed on
$\Sigma$ instead of on $A$.

**Solution.**

*Step 1.* $\lVert A \rVert_F^2 = \operatorname{tr}(A^{\top}A)$ by definition.

*Step 2.* Substitute the SVD: $A^{\top}A = V\Sigma^{\top}\Sigma V^{\top}$.

*Step 3.* The trace is cyclic, so
$\operatorname{tr}(V \Sigma^{\top}\Sigma V^{\top}) = \operatorname{tr}(\Sigma^{\top}\Sigma V^{\top}V) = \operatorname{tr}(\Sigma^{\top}\Sigma)$.

*Step 4.* $\Sigma^{\top}\Sigma = \operatorname{diag}(\sigma_1^2, \dots, \sigma_r^2, 0, \dots, 0)$.

$$
\boxed{\lVert A \rVert_F^2 = \sum_{i=1}^{r}\sigma_i^2}
$$

**Key takeaway.** This is statement 4 of Theorem 4.4, and it is what makes "energy retained" in a
truncation a statement about singular values.

In [10]:
A = rng.standard_normal((6, 4))
s = np.linalg.svd(A, compute_uv=False)
print("||A||_F^2      :", np.linalg.norm(A) ** 2)
print("sum sigma_i^2  :", (s ** 2).sum())
print("trace(A^T A)   :", np.trace(A.T @ A))
assert abs(np.linalg.norm(A) ** 2 - (s ** 2).sum()) < 1e-12

||A||_F^2      : 16.044935452954824
sum sigma_i^2  : 16.044935452954824
trace(A^T A)   : 16.044935452954824


### Problem L1.2 — Singular vectors as eigenvectors

**Statement.** Let $A = U\Sigma V^{\top}$. Prove that $v_i$ is an eigenvector of $A^{\top}A$ and
$u_i$ an eigenvector of $AA^{\top}$, both with eigenvalue $\sigma_i^2$.

**Intuition.** Forming $A^{\top}A$ cancels $U$ and forming $AA^{\top}$ cancels $V$.

**Solution.**

*Step 1.* $A^{\top}A = V\Sigma^{\top}U^{\top}U\Sigma V^{\top} = V(\Sigma^{\top}\Sigma)V^{\top}$.

*Step 2.* Multiply on the right by $V$: $(A^{\top}A)V = V(\Sigma^{\top}\Sigma)$, whose $i$-th
column reads $(A^{\top}A)v_i = \sigma_i^2 v_i$.

*Step 3.* Symmetrically $AA^{\top} = U(\Sigma\Sigma^{\top})U^{\top}$, giving
$(AA^{\top})u_i = \sigma_i^2 u_i$.

$$
\boxed{(A^{\top}A)v_i = \sigma_i^2 v_i, \qquad (AA^{\top})u_i = \sigma_i^2 u_i}
$$

**Key takeaway.** $A^{\top}A$ and $AA^{\top}$ have the same non-zero spectrum; the two Gram
matrices differ only in how many zeros they carry.

In [11]:
A = rng.standard_normal((5, 3))
U, s, Vt = np.linalg.svd(A, full_matrices=False)
for i in range(3):
    r1 = np.linalg.norm(A.T @ A @ Vt[i] - s[i] ** 2 * Vt[i])
    r2 = np.linalg.norm(A @ A.T @ U[:, i] - s[i] ** 2 * U[:, i])
    print(f"i={i}  sigma^2 = {s[i] ** 2:8.4f}   residuals {r1:.2e}, {r2:.2e}")
    assert r1 < 1e-12 and r2 < 1e-12
print("nonzero spectrum of A^T A:", np.sort(np.linalg.eigvalsh(A.T @ A))[::-1])
print("nonzero spectrum of A A^T:", np.sort(np.linalg.eigvalsh(A @ A.T))[::-1][:3])

i=0  sigma^2 =  10.6912   residuals 9.48e-15, 8.56e-15
i=1  sigma^2 =   4.4665   residuals 3.93e-15, 5.19e-15
i=2  sigma^2 =   1.5633   residuals 5.69e-16, 5.38e-16
nonzero spectrum of A^T A: [10.6912  4.4665  1.5633]
nonzero spectrum of A A^T: [10.6912  4.4665  1.5633]


### Problem L1.3 — Full SVD of a rank-one $2 \times 2$

**Statement.** Compute the full SVD of $A = \begin{pmatrix} 3 & 0 \\ 4 & 0 \end{pmatrix}$.

**Intuition.** The matrix sends $e_1$ to $(3,4)$ and kills $e_2$, so one singular value is
$\lVert (3,4) \rVert = 5$ and the other is $0$.

**Solution.**

*Step 1.* $A^{\top}A = \begin{pmatrix} 25 & 0 \\ 0 & 0\end{pmatrix}$, so $\sigma_1 = 5$,
$\sigma_2 = 0$.

*Step 2.* The eigenvectors of $A^{\top}A$ are $v_1 = e_1$ and $v_2 = e_2$, so $V = I_2$.

*Step 3.* $u_1 = Av_1/\sigma_1 = \tfrac15(3,4)^{\top}$, and $u_2 = \tfrac15(-4,3)^{\top}$
completes the orthonormal basis.

$$
\boxed{U = \frac{1}{5}\begin{pmatrix} 3 & -4 \\ 4 & 3\end{pmatrix}, \quad \Sigma = \begin{pmatrix} 5 & 0 \\ 0 & 0\end{pmatrix}, \quad V = I_2}
$$

**Key takeaway.** The second left singular vector spans $\operatorname{Null}(A^{\top})$; it carries
no information about $A$, only about the orthogonal complement of its column space.

In [12]:
A = np.array([[3.0, 0.0], [4.0, 0.0]])
Uh = np.array([[3.0, -4.0], [4.0, 3.0]]) / 5.0
Sh = np.diag([5.0, 0.0])
Vh = np.eye(2)
print("hand-built residual:", np.linalg.norm(A - Uh @ Sh @ Vh.T))
print("||U^T U - I||      :", np.linalg.norm(Uh.T @ Uh - np.eye(2)))
print("singular values    :", np.linalg.svd(A, compute_uv=False))
assert np.linalg.norm(A - Uh @ Sh @ Vh.T) < 1e-14
assert np.allclose(np.linalg.svd(A, compute_uv=False), [5.0, 0.0])

hand-built residual: 0.0
||U^T U - I||      : 3.7682219008410604e-17
singular values    : [5. 0.]


### Problem L1.4 — SVD of a real symmetric matrix

**Statement.** Let $A = Q\Lambda Q^{\top}$ be symmetric. Express an SVD of $A$ in terms of $Q$ and
$\Lambda$.

**Intuition.** Singular values must be non-negative, so the signs of the eigenvalues have to be
absorbed into one of the orthogonal factors.

**Solution.**

*Step 1.* Write $s_i = \operatorname{sgn}(\lambda_i)$ with $\operatorname{sgn}(0) = 1$, and
$S = \operatorname{diag}(s_1, \dots, s_n)$, so $\Lambda = S \lvert \Lambda \rvert$ where
$\lvert \Lambda \rvert = \operatorname{diag}(\lvert\lambda_1\rvert, \dots)$.

*Step 2.* Then $A = Q S \lvert \Lambda \rvert Q^{\top} = (QS)\,\lvert\Lambda\rvert\,Q^{\top}$.

*Step 3.* $S$ is orthogonal, so $QS$ is orthogonal. Sorting $\lvert\lambda_i\rvert$ into
descending order by a permutation applied to all three factors puts it in the form of
Definition 3.4.

$$
\boxed{U = QS, \qquad \Sigma = \lvert \Lambda \rvert, \qquad V = Q, \qquad \sigma_i = \lvert \lambda_i \rvert}
$$

**Key takeaway.** For symmetric $A$ the singular values are $\lvert \lambda_i \rvert$; they agree
with the eigenvalues exactly when $A \succeq 0$.

In [13]:
M = rng.standard_normal((4, 4))
A = (M + M.T) / 2
lam, Q = np.linalg.eigh(A)
S = np.diag(np.where(lam >= 0, 1.0, -1.0))
order = np.argsort(np.abs(lam))[::-1]
Uh = (Q @ S)[:, order]
Sig = np.diag(np.abs(lam)[order])
Vh = Q[:, order]
print("eigenvalues     :", lam)
print("|eigenvalues|   :", np.sort(np.abs(lam))[::-1])
print("singular values :", np.linalg.svd(A, compute_uv=False))
print("residual        :", np.linalg.norm(A - Uh @ Sig @ Vh.T))
assert np.allclose(np.sort(np.abs(lam))[::-1], np.linalg.svd(A, compute_uv=False))
assert np.linalg.norm(A - Uh @ Sig @ Vh.T) < 1e-13

eigenvalues     : [-0.9808  0.613   2.0938  3.2726]
|eigenvalues|   : [3.2726 2.0938 0.9808 0.613 ]
singular values : [3.2726 2.0938 0.9808 0.613 ]
residual        : 1.0594488999422677e-15


### Problem L1.5 — Truncation error in both norms

**Statement.** With $A_k = \sum_{i=1}^{k}\sigma_i u_i v_i^{\top}$, prove
$\lVert A - A_k \rVert_{\mathrm{op}} = \sigma_{k+1}$ and
$\lVert A - A_k \rVert_F = \bigl(\sum_{i \gt k}\sigma_i^2\bigr)^{1/2}$.

**Intuition.** The residual is itself in SVD form, so its norms can be read off directly.

**Solution.**

*Step 1.* $E_k = A - A_k = \sum_{i \gt k}\sigma_i u_i v_i^{\top}$.

*Step 2.* The vectors $u_{k+1}, \dots, u_r$ are orthonormal, as are $v_{k+1}, \dots, v_r$, so this
expression *is* an SVD of $E_k$ with singular values $\sigma_{k+1} \ge \dots \ge \sigma_r$.

*Step 3.* Apply Theorem 4.4 statement 4 to $E_k$.

$$
\boxed{\lVert A - A_k \rVert_{\mathrm{op}} = \sigma_{k+1}, \qquad \lVert A - A_k \rVert_F = \Bigl( \sum_{i \gt k}\sigma_i^2 \Bigr)^{1/2}}
$$

**Key takeaway.** This is only the *value* at $A_k$. That no other rank-$k$ matrix does better is
the separate content of Theorem 4.6, proved in Proof 5.7.

In [14]:
A = rng.standard_normal((7, 5))
U, s, Vt = np.linalg.svd(A, full_matrices=False)
for k in range(1, 5):
    Ak = (U[:, :k] * s[:k]) @ Vt[:k]
    eo, ef = np.linalg.norm(A - Ak, 2), np.linalg.norm(A - Ak)
    print(f"k={k}  ||A-A_k||_op = {eo:.6f} (sigma_{k + 1} = {s[k]:.6f})"
          f"   ||A-A_k||_F = {ef:.6f} (tail = {np.sqrt((s[k:] ** 2).sum()):.6f})")
    assert abs(eo - s[k]) < 1e-12
    assert abs(ef - np.sqrt((s[k:] ** 2).sum())) < 1e-12

k=1  ||A-A_k||_op = 3.363207 (sigma_2 = 3.363207)   ||A-A_k||_F = 4.454871 (tail = 4.454871)
k=2  ||A-A_k||_op = 2.271989 (sigma_3 = 2.271989)   ||A-A_k||_F = 2.921424 (tail = 2.921424)
k=3  ||A-A_k||_op = 1.456570 (sigma_4 = 1.456570)   ||A-A_k||_F = 1.836515 (tail = 1.836515)
k=4  ||A-A_k||_op = 1.118566 (sigma_5 = 1.118566)   ||A-A_k||_F = 1.118566 (tail = 1.118566)


### Problem L1.6 — Pseudoinverse of a full-column-rank matrix

**Statement.** Compute $A^{+}$ for
$A = \begin{pmatrix} 1 & 0 \\ 1 & 1 \\ 0 & 1 \end{pmatrix}$.

**Intuition.** With full column rank, $A^{\top}A$ is invertible and $A^{+} = (A^{\top}A)^{-1}A^{\top}$.

**Solution.**

*Step 1.* $A^{\top}A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}$, with determinant $3$.

*Step 2.* $(A^{\top}A)^{-1} = \tfrac13\begin{pmatrix} 2 & -1 \\ -1 & 2\end{pmatrix}$.

*Step 3.* Multiply by $A^{\top}$:

$$
A^{+} = \frac13\begin{pmatrix} 2 & -1 \\ -1 & 2\end{pmatrix}\begin{pmatrix} 1 & 1 & 0 \\ 0 & 1 & 1\end{pmatrix}
= \frac13 \begin{pmatrix} 2 & 1 & -1 \\ -1 & 1 & 2 \end{pmatrix} .
$$

*Step 4.* Check that this agrees with $V\Sigma^{+}U^{\top}$: both satisfy the four Penrose
conditions, and Theorem 4.7 says there is only one such matrix.

$$
\boxed{A^{+} = \frac13\begin{pmatrix} 2 & 1 & -1 \\ -1 & 1 & 2\end{pmatrix}}
$$

**Key takeaway.** Full column rank makes $A^{+}A = I_2$, so $A^{+}$ is a left inverse; it is never
a right inverse unless $A$ is square.

In [15]:
A = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0]])
Ap_hand = np.array([[2.0, 1.0, -1.0], [-1.0, 1.0, 2.0]]) / 3.0
print("A^+ (hand)  :\n", Ap_hand)
print("||hand - pinv||:", np.linalg.norm(Ap_hand - np.linalg.pinv(A)))
print("A^+ A       :\n", Ap_hand @ A)
print("A A^+       :\n", A @ Ap_hand)
assert np.allclose(Ap_hand, np.linalg.pinv(A))
assert np.allclose(Ap_hand @ A, np.eye(2))
assert not np.allclose(A @ Ap_hand, np.eye(3))

A^+ (hand)  :
 [[ 0.6667  0.3333 -0.3333]
 [-0.3333  0.3333  0.6667]]
||hand - pinv||: 3.805652793121308e-16
A^+ A       :
 [[1. 0.]
 [0. 1.]]
A A^+       :
 [[ 0.6667  0.3333 -0.3333]
 [ 0.3333  0.6667  0.3333]
 [-0.3333  0.3333  0.6667]]


### Problem L1.7 — Minimum-norm solution of an underdetermined system

**Statement.** Find the minimum 2-norm solution of $Ax = b$ with
$A = \begin{pmatrix} 1 & 1 \end{pmatrix}$ and $b = [2]$.

**Intuition.** The solutions form a line; the pseudoinverse picks the point of that line closest to
the origin, which is where the line meets $\operatorname{Col}(A^{\top})$.

**Solution.**

*Step 1.* $AA^{\top} = [2]$, so $A$ has full row rank and $A^{+} = A^{\top}(AA^{\top})^{-1}$.

*Step 2.* $A^{+} = \tfrac12 (1,1)^{\top}$.

*Step 3.* $x^{\star} = A^{+}b = (1,1)^{\top}$.

*Step 4.* The general solution is $x = (2-t, t)^{\top}$ with
$\lVert x \rVert^2 = (2-t)^2 + t^2 = 2t^2 - 4t + 4$, minimized at $t = 1$.

$$
\boxed{x^{\star} = \begin{pmatrix} 1 \\ 1 \end{pmatrix}, \qquad \lVert x^{\star} \rVert_2 = \sqrt2}
$$

**Key takeaway.** Theorem 4.7 puts $x^{\star}$ in $\operatorname{Col}(A^{\top})$; here that is the
line $x_1 = x_2$, and the intersection with the solution set is a single point.

In [16]:
A = np.array([[1.0, 1.0]])
b = np.array([2.0])
xstar = np.linalg.pinv(A) @ b
print("x* =", xstar, "  ||x*|| =", np.linalg.norm(xstar), " sqrt2 =", np.sqrt(2))
ts = np.linspace(-1.0, 3.0, 9)
norms = [np.linalg.norm([2 - t, t]) for t in ts]
print("t      :", np.round(ts, 2))
print("||x(t)||:", np.round(norms, 4))
assert np.allclose(xstar, [1.0, 1.0])
assert abs(min(norms) - np.sqrt(2)) < 1e-12

x* = [1. 1.]   ||x*|| = 1.4142135623730945  sqrt2 = 1.4142135623730951
t      : [-1.  -0.5  0.   0.5  1.   1.5  2.   2.5  3. ]
||x(t)||: [3.1623 2.5495 2.     1.5811 1.4142 1.5811 2.     2.5495 3.1623]


### Problem L1.8 — Pseudoinverse of a scaled rank-one matrix

**Statement.** Let $A = c\,uv^{\top}$ with $u \in \mathbb{R}^m$, $v \in \mathbb{R}^n$ unit vectors
and $c \neq 0$. Prove $A^{+} = c^{-1} v u^{\top}$.

**Intuition.** The compact SVD is $A = \lvert c \rvert \,(\operatorname{sgn}(c)u)\,v^{\top}$, so
inverting means reciprocating $c$ and swapping the vectors.

**Solution.** Set $X = c^{-1}vu^{\top}$ and verify Definition 3.6, using
$u^{\top}u = v^{\top}v = 1$.

*Step 1.* $AXA = c\,u v^{\top} \cdot c^{-1} v u^{\top} \cdot c \, u v^{\top} = c\,uv^{\top} = A$.

*Step 2.* $XAX = c^{-1}vu^{\top} \cdot c\,uv^{\top} \cdot c^{-1}vu^{\top} = c^{-1}vu^{\top} = X$.

*Step 3.* $AX = uu^{\top}$, symmetric.

*Step 4.* $XA = vv^{\top}$, symmetric.

By the uniqueness half of Theorem 4.7, $X$ is *the* pseudoinverse.

$$
\boxed{A^{+} = \frac{1}{c}\, v u^{\top}}
$$

**Key takeaway.** $AA^{+} = uu^{\top}$ and $A^{+}A = vv^{\top}$ are the orthogonal projectors onto
$\operatorname{Col}(A)$ and $\operatorname{Col}(A^{\top})$, as Theorem 4.7 predicts.

In [17]:
u = rng.standard_normal(4); u /= np.linalg.norm(u)
v = rng.standard_normal(3); v /= np.linalg.norm(v)
c = -2.5
A = c * np.outer(u, v)
X = np.outer(v, u) / c
print("||X - pinv(A)||_F :", np.linalg.norm(X - np.linalg.pinv(A)))
print("||A X - u u^T||_F :", np.linalg.norm(A @ X - np.outer(u, u)))
print("||X A - v v^T||_F :", np.linalg.norm(X @ A - np.outer(v, v)))
assert np.linalg.norm(X - np.linalg.pinv(A)) < 1e-12
assert np.linalg.norm(A @ X - np.outer(u, u)) < 1e-12

||X - pinv(A)||_F : 2.411951935313947e-16
||A X - u u^T||_F : 1.2794688166302258e-16
||X A - v v^T||_F : 1.642041997679149e-16


### Problem L1.9 — Polar decomposition of an invertible matrix

**Statement.** Find $A = QH$ for $A = \begin{pmatrix} 0 & -2 \\ 1 & 0\end{pmatrix}$.

**Intuition.** $H = (A^{\top}A)^{1/2}$ is the pure stretch; $Q = AH^{-1}$ is what is left, and it
must be orthogonal.

**Solution.**

*Step 1.* $A^{\top}A = \begin{pmatrix} 1 & 0 \\ 0 & 4\end{pmatrix}$.

*Step 2.* It is diagonal with positive entries, so
$H = \operatorname{diag}(1, 2)$.

*Step 3.* $Q = AH^{-1} = \begin{pmatrix} 0 & -2 \\ 1 & 0\end{pmatrix}\operatorname{diag}(1, \tfrac12) = \begin{pmatrix} 0 & -1 \\ 1 & 0\end{pmatrix}$,
and $Q^{\top}Q = I$.

$$
\boxed{Q = \begin{pmatrix} 0 & -1 \\ 1 & 0\end{pmatrix}, \qquad H = \begin{pmatrix} 1 & 0 \\ 0 & 2\end{pmatrix}}
$$

**Key takeaway.** $A$ is a quarter-turn preceded by a stretch of factor $2$ along $e_2$; the
singular values $2$ and $1$ are the diagonal of $H$.

In [18]:
A = np.array([[0.0, -2.0], [1.0, 0.0]])
Qh = np.array([[0.0, -1.0], [1.0, 0.0]])
Hh = np.diag([1.0, 2.0])
print("||Q H - A||_F   :", np.linalg.norm(Qh @ Hh - A))
print("||Q^T Q - I||_F :", np.linalg.norm(Qh.T @ Qh - np.eye(2)))
print("singular values :", np.linalg.svd(A, compute_uv=False))
U, s, Vt = np.linalg.svd(A)
print("library Q, H    :\n", U @ Vt, "\n", Vt.T @ np.diag(s) @ Vt)
assert np.linalg.norm(Qh @ Hh - A) < 1e-14
assert np.allclose(U @ Vt, Qh) and np.allclose(Vt.T @ np.diag(s) @ Vt, Hh)

||Q H - A||_F   : 0.0
||Q^T Q - I||_F : 0.0
singular values : [2. 1.]
library Q, H    :
 [[ 0. -1.]
 [ 1.  0.]] 
 [[1. 0.]
 [0. 2.]]


### Problem L1.10 — Polar decomposition of a singular matrix

**Statement.** Find a polar decomposition of $A = \begin{pmatrix} 1 & 1 \\ 1 & 1\end{pmatrix}$, and
say whether it is unique.

**Intuition.** $A$ is already symmetric positive semidefinite, so it is its own stretch factor.

**Solution.**

*Step 1.* $A^{\top}A = \begin{pmatrix} 2 & 2 \\ 2 & 2 \end{pmatrix} = 2A$.

*Step 2.* $A = 2\,v_1v_1^{\top}$ with $v_1 = \tfrac{1}{\sqrt2}(1,1)^{\top}$, so
$A^{\top}A = 4 v_1v_1^{\top}$ and $H = (A^{\top}A)^{1/2} = 2 v_1v_1^{\top} = A$.

*Step 3.* $Q = I$ then gives $QH = A$.

*Step 4.* Uniqueness. $\operatorname{rank}(A) = 1 \lt 2$, so by Theorem 4.8 the factor $Q$ is
*not* unique: with $v_2 = \tfrac1{\sqrt2}(1,-1)^{\top}$, the matrix
$Q' = v_1v_1^{\top} - v_2v_2^{\top} = \left[\begin{smallmatrix} 0 & 1 \\ 1 & 0\end{smallmatrix}\right]$
is also orthogonal and also satisfies $Q'H = A$.

$$
\boxed{H = A = \begin{pmatrix} 1 & 1 \\ 1 & 1\end{pmatrix}, \qquad Q = I \ \text{ or } \ \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}}
$$

**Key takeaway.** $H$ is always unique; $Q$ is unique exactly at full column rank, and the freedom
lives on $\operatorname{Null}(A)$.

In [19]:
A = np.ones((2, 2))
Hh = A.copy()
for Q in (np.eye(2), np.array([[0.0, 1.0], [1.0, 0.0]])):
    print("Q =\n", Q, "\n  Q^T Q - I:", np.linalg.norm(Q.T @ Q - np.eye(2)),
          "  ||Q H - A||:", np.linalg.norm(Q @ Hh - A))
    assert np.linalg.norm(Q.T @ Q - np.eye(2)) < 1e-14
    assert np.linalg.norm(Q @ Hh - A) < 1e-14
print("rank A =", np.linalg.matrix_rank(A), " -> Q not unique (Theorem 4.8)")
print("||H^2 - A^T A||:", np.linalg.norm(Hh @ Hh - A.T @ A))
assert np.linalg.norm(Hh @ Hh - A.T @ A) < 1e-14

Q =
 [[1. 0.]
 [0. 1.]] 
  Q^T Q - I: 0.0   ||Q H - A||: 0.0
Q =
 [[0. 1.]
 [1. 0.]] 
  Q^T Q - I: 0.0   ||Q H - A||: 0.0
rank A = 1  -> Q not unique (Theorem 4.8)
||H^2 - A^T A||: 0.0


### Problem L1.11 — Polar decomposition of a reflection with stretch

**Statement.** Find $A = QH$ for $A = \begin{pmatrix} -3 & 0 \\ 0 & 2\end{pmatrix}$.

**Intuition.** The negative entry is a reflection; it belongs to $Q$, not to $H$, because $H$ must
be positive semidefinite.

**Solution.**

*Step 1.* $A^{\top}A = \operatorname{diag}(9, 4)$, so $H = \operatorname{diag}(3, 2)$.

*Step 2.* $Q = AH^{-1} = \operatorname{diag}(-3, 2)\operatorname{diag}(\tfrac13, \tfrac12) = \operatorname{diag}(-1, 1)$.

*Step 3.* $\det Q = -1$: this $Q$ is orthogonal but is a reflection, not a rotation.

$$
\boxed{Q = \begin{pmatrix} -1 & 0 \\ 0 & 1\end{pmatrix}, \qquad H = \begin{pmatrix} 3 & 0 \\ 0 & 2\end{pmatrix}}
$$

**Key takeaway.** Polar decomposition produces an *orthogonal* factor, which may have determinant
$-1$. Applications needing a proper rotation must add the determinant correction of Problem L2.10.

In [20]:
A = np.diag([-3.0, 2.0])
Qh, Hh = np.diag([-1.0, 1.0]), np.diag([3.0, 2.0])
print("||Q H - A||   :", np.linalg.norm(Qh @ Hh - A))
print("det Q         :", np.linalg.det(Qh))
print("singular values:", np.linalg.svd(A, compute_uv=False))
assert np.linalg.norm(Qh @ Hh - A) < 1e-14
assert np.isclose(np.linalg.det(Qh), -1.0)
assert np.allclose(np.linalg.svd(A, compute_uv=False), [3.0, 2.0])

||Q H - A||   : 0.0
det Q         : -1.0
singular values: [3. 2.]


### Problem L1.12 — Jordan form of a defective $2 \times 2$

**Statement.** Find $J$ for $A = \begin{pmatrix} 4 & 1 \\ -1 & 2\end{pmatrix}$.

**Intuition.** A repeated eigenvalue with only one eigenvector forces a superdiagonal $1$.

**Solution.**

*Step 1.* $p_A(z) = (4-z)(2-z) + 1 = z^2 - 6z + 9 = (z-3)^2$, so $\lambda = 3$ with $m_3 = 2$.

*Step 2.* $A - 3I = \begin{pmatrix} 1 & 1 \\ -1 & -1\end{pmatrix}$ has rank $1$, so $r_1 = 1$.

*Step 3.* By Theorem 4.2, $b_{\ge 1} = r_0 - r_1 = 1$ block, and $b_{\ge 2} = r_1 - r_2 = 1 - 0 = 1$,
so that block has size $2$.

$$
\boxed{J = \begin{pmatrix} 3 & 1 \\ 0 & 3\end{pmatrix}}
$$

**Key takeaway.** $g_\lambda$ counts the blocks; here $g_3 = 1 \lt m_3 = 2$, which is the
definition of defective.

In [21]:
A = np.array([[4.0, 1.0], [-1.0, 2.0]])
N = A - 3 * np.eye(2)
ranks = [int(np.linalg.matrix_rank(np.linalg.matrix_power(N, j))) for j in range(3)]
print("characteristic polynomial:", np.poly(A), " -> z^2 - 6z + 9")
print("eigenvalues              :", np.linalg.eigvals(A))
print("rank (A-3I)^j            :", ranks)
print("blocks of size >= j      :", [ranks[j - 1] - ranks[j] for j in range(1, 3)])
assert np.allclose(np.poly(A), [1.0, -6.0, 9.0])
assert ranks == [2, 1, 0]

characteristic polynomial: [ 1. -6.  9.]  -> z^2 - 6z + 9
eigenvalues              : [3. 3.]
rank (A-3I)^j            : [2, 1, 0]
blocks of size >= j      : [1, 1]


### Problem L1.13 — Building a Jordan chain

**Statement.** For $A = \begin{pmatrix} 2 & 1 \\ 0 & 2 \end{pmatrix}$ construct a Jordan chain
$(x_1, x_2)$ and the matrix $P$ with $P^{-1}AP = J$.

**Intuition.** Definition 3.3: $x_1$ is a genuine eigenvector and $x_2$ is pulled back one step by
$A - \lambda I$.

**Solution.**

*Step 1.* $\lambda = 2$ and $N = A - 2I = \left[\begin{smallmatrix}0 & 1 \\ 0 & 0\end{smallmatrix}\right]$.

*Step 2.* Solve $Nx_1 = 0$: the second coordinate must vanish, so $x_1 = e_1$.

*Step 3.* Solve $Nx_2 = x_1$: the second coordinate of $x_2$ must be $1$, and the first is free;
take $x_2 = e_2$.

*Step 4.* $P = [\,x_1 \ x_2\,] = I_2$, and indeed $P^{-1}AP = A = J_2(2)$.

$$
\boxed{x_1 = e_1, \quad x_2 = e_2, \quad P = I_2, \quad J = \begin{pmatrix} 2 & 1 \\ 0 & 2\end{pmatrix}}
$$

**Key takeaway.** Proof 5.1 Step 3 builds every chain this way: start at the *top*, from a vector
outside $\operatorname{Null}(N^{k-1})$, and apply $N$ repeatedly to descend.

In [22]:
A = np.array([[2.0, 1.0], [0.0, 2.0]])
N = A - 2 * np.eye(2)
x1, x2 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
print("N x1 =", N @ x1, " (must be zero)")
print("N x2 =", N @ x2, " (must equal x1)")
P = np.column_stack([x1, x2])
print("P^-1 A P =\n", np.linalg.solve(P, A @ P))
assert np.allclose(N @ x1, 0.0) and np.allclose(N @ x2, x1)
assert np.allclose(np.linalg.solve(P, A @ P), A)

N x1 = [0. 0.]  (must be zero)
N x2 = [1. 0.]  (must equal x1)
P^-1 A P =
 [[2. 1.]
 [0. 2.]]


### Problem L1.14 — Schur form of a symmetric matrix is diagonal

**Statement.** Let $A \in \mathbb{R}^{n \times n}$ be symmetric. Show that its Schur factor is
diagonal, and justify the use of a *real orthogonal* $U$.

**Intuition.** A matrix that is both upper triangular and symmetric has nothing off the diagonal.

**Solution.**

*Step 1 — the real form is available.* Schur triangularization
([Module 06](../06_eigenvalues_eigenvectors_spectral_theory/first_principles.ipynb), Theorem 4.1)
is stated over $\mathbb{C}$. For symmetric $A$ every eigenvalue is real
(Module 06, Proof 5.2 Step 1), so at each stage of that induction the eigenvector may be taken
real, the Gram-Schmidt extension is real, and $U$ comes out real orthogonal. This is exactly Step 3
of Module 06, Proof 5.2; the real case is not an extra assumption.

*Step 2.* Write $A = UTU^{\top}$, so $T = U^{\top}AU$.

*Step 3.* $T^{\top} = (U^{\top}AU)^{\top} = U^{\top}A^{\top}U = U^{\top}AU = T$, so $T$ is
symmetric.

*Step 4.* $T$ is upper triangular and symmetric, so $t_{ij} = t_{ji} = 0$ whenever $i \neq j$.

$$
\boxed{T = \Lambda = \operatorname{diag}(\lambda_1, \dots, \lambda_n)}
$$

**Key takeaway.** This is the deduction of the spectral theorem from Schur; the off-diagonal entry
of Example 6.1 of Module 06 is precisely what symmetry removes.

In [23]:
M = rng.standard_normal((5, 5))
A = (M + M.T) / 2
lam, Q = np.linalg.eigh(A)
T = Q.T @ A @ Q
print("off-diagonal Frobenius mass of U^T A U:", np.linalg.norm(T - np.diag(np.diag(T))))
print("diagonal                              :", np.diag(T))
print("eigenvalues                           :", lam)
assert np.linalg.norm(T - np.diag(np.diag(T))) < 1e-13
assert np.allclose(np.sort(np.diag(T)), np.sort(lam))

off-diagonal Frobenius mass of U^T A U: 1.4274544226062786e-15
diagonal                              : [-3.0706 -1.4657  0.0772  0.7084  1.4283]
eigenvalues                           : [-3.0706 -1.4657  0.0772  0.7084  1.4283]


### Problem L1.15 — Real Schur form of a $2 \times 2$

**Statement.** Find a real orthogonal $U$ and upper triangular $T$ with $A = UTU^{\top}$ for
$A = \begin{pmatrix} 5 & -2 \\ 4 & -1\end{pmatrix}$.

**Intuition.** Take the first column of $U$ to be a unit eigenvector; the deflation step of
Module 06, Proof 5.1 then puts a zero below the corner.

**Solution.**

*Step 1.* $p_A(z) = (5-z)(-1-z) + 8 = z^2 - 4z + 3 = (z-3)(z-1)$, so $\lambda_1 = 3$,
$\lambda_2 = 1$: both real, so a real Schur form exists.

*Step 2.* $(A - 3I)v = 0$ with $A - 3I = \begin{pmatrix} 2 & -2 \\ 4 & -4\end{pmatrix}$ gives
$v \propto (1,1)$, so $u_1 = \tfrac{1}{\sqrt2}(1,1)^{\top}$.

*Step 3.* $u_2 = \tfrac{1}{\sqrt2}(-1,1)^{\top}$ completes the basis, so
$U = \tfrac1{\sqrt2}\left[\begin{smallmatrix}1 & -1 \\ 1 & 1\end{smallmatrix}\right]$.

*Step 4.* $T = U^{\top}AU = \begin{pmatrix} 3 & -6 \\ 0 & 1\end{pmatrix}$.

$$
\boxed{U = \frac{1}{\sqrt2}\begin{pmatrix} 1 & -1 \\ 1 & 1\end{pmatrix}, \qquad T = \begin{pmatrix} 3 & -6 \\ 0 & 1\end{pmatrix}}
$$

**Key takeaway.** The entry $-6$ is the obstruction to diagonality; it survives because $A$ is not
symmetric, exactly as Problem L1.14 predicts.

In [24]:
A = np.array([[5.0, -2.0], [4.0, -1.0]])
U = np.array([[1.0, -1.0], [1.0, 1.0]]) / np.sqrt(2)
T = U.T @ A @ U
print("eigenvalues:", np.sort(np.linalg.eigvals(A).real))
print("U^T U - I  :", np.linalg.norm(U.T @ U - np.eye(2)))
print("T          :\n", T)
assert np.allclose(T, np.array([[3.0, -6.0], [0.0, 1.0]]))
assert np.linalg.norm(A - U @ T @ U.T) < 1e-14

eigenvalues: [1. 3.]
U^T U - I  : 3.1560822113208575e-16
T          :
 [[ 3. -6.]
 [ 0.  1.]]


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Principal components from the SVD of the data matrix

**Statement.** Let $X \in \mathbb{R}^{N \times d}$ be column-centred with SVD $X = U\Sigma V^{\top}$.
Show that the principal directions are the columns of $V$ and that the explained variances are
$\sigma_i^2/(N-1)$.

**Intuition.** The sample covariance is $X^{\top}X$ up to a constant, and Theorem 4.4 says the
right singular vectors already diagonalize it.

**Solution.**

*Step 1.* The sample covariance is $S = \dfrac{1}{N-1}X^{\top}X$.

*Step 2.* Substitute the SVD:

$$
S = \frac{1}{N-1}V\Sigma^{\top}\Sigma V^{\top} = V \Bigl( \frac{\Sigma^{\top}\Sigma}{N-1} \Bigr) V^{\top} .
$$

*Step 3.* $V$ is orthogonal and the middle factor is diagonal with non-negative entries, so this is
the spectral decomposition of $S$.

$$
\boxed{S v_i = \frac{\sigma_i^2}{N-1}\, v_i, \qquad \text{principal directions } v_1, \dots, v_d}
$$

**Key takeaway.** Taking the SVD of $X$ never forms $X^{\top}X$, whose condition number is the
square of $X$'s; Section 8.1 of the theory notebook measures a factor $10^{7}$ difference in the
smallest singular value.

In [25]:
N, d = 300, 4
Xd = rng.standard_normal((N, 2)) @ rng.standard_normal((2, d)) + 0.3 * rng.standard_normal((N, d))
Xd -= Xd.mean(axis=0)
U, s, Vt = np.linalg.svd(Xd, full_matrices=False)
S_cov = Xd.T @ Xd / (N - 1)
lam, Q = np.linalg.eigh(S_cov)
print("sigma_i^2/(N-1)      :", s ** 2 / (N - 1))
print("eigenvalues of S     :", lam[::-1])
print("|cos angle(v_i, q_i)| :", np.abs(np.diag(Vt @ Q[:, ::-1])))
assert np.allclose(s ** 2 / (N - 1), lam[::-1])
assert np.allclose(np.abs(np.diag(Vt @ Q[:, ::-1])), 1.0)

sigma_i^2/(N-1)      : [3.6205 2.7057 0.0915 0.0839]
eigenvalues of S     : [3.6205 2.7057 0.0915 0.0839]
|cos angle(v_i, q_i)| : [1. 1. 1. 1.]


### Problem L2.2 — Latent semantic analysis coordinates

**Statement.** A term-document matrix $A \in \mathbb{R}^{m \times n}$ is truncated to
$A_k = U_k\Sigma_kV_k^{\top}$. Give the $k$-dimensional coordinates of document $j$ and show they
cost no extra work once the SVD is known.

**Intuition.** $U_k$ is an orthonormal basis for the retained topic space, so projecting a document
onto it is one multiplication by $U_k^{\top}$.

**Solution.**

*Step 1.* Document $j$ is the column $a_j = Ae_j$, and its rank-$k$ image is
$A_ke_j = U_k\Sigma_kV_k^{\top}e_j$.

*Step 2.* Write $\tilde{v}_j = V_k^{\top}e_j \in \mathbb{R}^k$, the $j$-th row of $V_k$ written as a
column.

*Step 3.* Project onto the topic basis using $U_k^{\top}U_k = I_k$:

$$
U_k^{\top}(A_k e_j) = U_k^{\top}U_k \Sigma_k \tilde{v}_j = \Sigma_k \tilde{v}_j .
$$

$$
\boxed{\tilde{a}_j = U_k^{\top}A_k e_j = \Sigma_k V_k^{\top} e_j}
$$

**Key takeaway.** The coordinates are the rows of $V_k$ scaled by the singular values; the topic
embedding of a corpus is a by-product of the SVD, not a second computation.

In [26]:
m_t, n_t, k_t = 40, 25, 5
Aterm = np.abs(rng.standard_normal((m_t, 6)) @ rng.standard_normal((6, n_t)))
Ut, st, Vtt = np.linalg.svd(Aterm, full_matrices=False)
Uk, Sk, Vk = Ut[:, :k_t], np.diag(st[:k_t]), Vtt[:k_t].T
Ak = Uk @ Sk @ Vk.T
coords_a = Uk.T @ Ak
coords_b = Sk @ Vk.T
print("||U_k^T A_k - Sigma_k V_k^T||_F :", np.linalg.norm(coords_a - coords_b))
print("coordinates of document 0       :", coords_b[:, 0])
print("||A - A_k||_F / ||A||_F         :", np.linalg.norm(Aterm - Ak) / np.linalg.norm(Aterm))
assert np.linalg.norm(coords_a - coords_b) < 1e-11

||U_k^T A_k - Sigma_k V_k^T||_F : 5.851393769118405e-14
coordinates of document 0       : [-16.9934  -4.5971  -3.824   -2.1749  -1.6951]
||A - A_k||_F / ||A||_F         : 0.34877444577723077


### Problem L2.3 — Low-rank adaptation: rank and norm bounds

**Statement.** A weight update is parameterized as $\Delta W = BC$ with
$B \in \mathbb{R}^{d \times r}$, $C \in \mathbb{R}^{r \times k}$ and $r \ll \min(d,k)$. Prove
$\operatorname{rank}(\Delta W) \le r$ and
$\lVert \Delta W \rVert_F \le \lVert B \rVert_F \lVert C \rVert_{\mathrm{op}}$, and say what
Theorem 4.6 adds.

**Intuition.** The product factors through an $r$-dimensional space, and the Frobenius norm is
sub-multiplicative against the operator norm.

**Solution.**

*Step 1 — rank.* $\operatorname{Col}(BC) \subseteq \operatorname{Col}(B)$, which has dimension at
most $r$.

*Step 2 — norm.* Writing $c_j$ for the columns of $C$,

$$
\lVert BC \rVert_F^2 = \sum_j \lVert Bc_j \rVert^2 \le \lVert B \rVert_{\mathrm{op}}^2 \sum_j \lVert c_j \rVert^2 = \lVert B \rVert_{\mathrm{op}}^2 \lVert C \rVert_F^2 ,
$$

and the same argument on rows gives
$\lVert BC \rVert_F \le \lVert B \rVert_F \lVert C \rVert_{\mathrm{op}}$. Since
$\lVert \cdot \rVert_{\mathrm{op}} \le \lVert \cdot \rVert_F$, the weaker bound
$\lVert B \rVert_F\lVert C \rVert_F$ follows.

*Step 3 — what is optimal.* By Theorem 4.6, among all rank-$r$ matrices the one closest to a target
$T$ is $T_r = \sum_{i \le r}\sigma_i u_iv_i^{\top}$, with error $\bigl(\sum_{i \gt r}\sigma_i^2\bigr)^{1/2}$.

$$
\boxed{\operatorname{rank}(\Delta W) \le r, \qquad \lVert \Delta W \rVert_F \le \lVert B \rVert_F \lVert C \rVert_{\mathrm{op}} \le \lVert B \rVert_F \lVert C \rVert_F}
$$

**Key takeaway.** Whether a rank-$r$ update can work is decided by the singular-value decay of the
update it is trying to imitate — a property of the problem, not of the parameterization. Note also
that the rank-$\le r$ set is not convex, so Theorem 4.6 bounds what is achievable, not what
gradient descent will find.

In [27]:
d_l, k_l, r_l = 30, 20, 4
Bm = rng.standard_normal((d_l, r_l))
Cm = rng.standard_normal((r_l, k_l))
dW = Bm @ Cm
print("rank(BC)                       :", np.linalg.matrix_rank(dW), " <= r =", r_l)
print("||BC||_F                       :", np.linalg.norm(dW))
print("||B||_F ||C||_op               :", np.linalg.norm(Bm) * np.linalg.norm(Cm, 2))
print("||B||_F ||C||_F                :", np.linalg.norm(Bm) * np.linalg.norm(Cm))
Target = rng.standard_normal((d_l, k_l))
Ut2, st2, Vt2 = np.linalg.svd(Target, full_matrices=False)
best = np.linalg.norm(Target - (Ut2[:, :r_l] * st2[:r_l]) @ Vt2[:r_l])
print("best rank-r error / tail       :", best, np.sqrt((st2[r_l:] ** 2).sum()))
assert np.linalg.matrix_rank(dW) <= r_l
assert np.linalg.norm(dW) <= np.linalg.norm(Bm) * np.linalg.norm(Cm, 2) + 1e-12
assert abs(best - np.sqrt((st2[r_l:] ** 2).sum())) < 1e-12

rank(BC)                       : 4  <= r = 4
||BC||_F                       : 43.15221381946235
||B||_F ||C||_op               : 57.032054680575406
||B||_F ||C||_F                : 89.16583358955079
best rank-r error / tail       : 17.84707638352413 17.84707638352413


### Problem L2.4 — Choosing the rank from retained energy

**Statement.** Express the fraction of Frobenius energy retained by $A_k$, and compute the smallest
$k$ reaching $95$ percent for a matrix with singular values $\sigma_i = 2^{-(i-1)/2}$,
$i = 1, \dots, 20$.

**Intuition.** "Energy" is $\lVert A \rVert_F^2$, which Problem L1.1 identifies with
$\sum_i \sigma_i^2$, so the question is about a geometric series.

**Solution.**

*Step 1.* $\lVert A_k \rVert_F^2 = \sum_{i \le k}\sigma_i^2$ and
$\lVert A \rVert_F^2 = \sum_{i \le r}\sigma_i^2$, so the retained fraction is

$$
\rho(k) = \frac{\sum_{i=1}^{k}\sigma_i^2}{\sum_{i=1}^{r}\sigma_i^2} .
$$

*Step 2.* Here $\sigma_i^2 = 2^{-(i-1)}$, so
$\sum_{i \le k}\sigma_i^2 = 2 - 2^{1-k}$ and $\sum_{i \le 20}\sigma_i^2 = 2 - 2^{-19}$.

*Step 3.* $\rho(k) \ge 0.95$ needs $2 - 2^{1-k} \ge 0.95(2 - 2^{-19})$, that is
$2^{1-k} \le 0.1 + 0.95 \cdot 2^{-19}$, hence $k \ge 1 + \log_2(1/0.1) = 4.32$.

$$
\boxed{\rho(k) = \frac{\sum_{i \le k}\sigma_i^2}{\sum_i \sigma_i^2}, \qquad k_{\min} = 5 \ \text{ with } \ \rho(5) = 0.9688}
$$

**Key takeaway.** Energy is a *squared* quantity, so it saturates about twice as fast as the
singular values themselves; a $95$ percent energy threshold is a much weaker requirement than a
$95$ percent reduction in $\sigma$.

In [28]:
sv = 2.0 ** (-np.arange(20) / 2.0)
energy = np.cumsum(sv ** 2) / (sv ** 2).sum()
kmin = int(np.argmax(energy >= 0.95)) + 1
for k in range(1, 8):
    print(f"k={k}  retained energy {energy[k - 1]:.4f}   relative error"
          f" {np.sqrt(1 - energy[k - 1]):.4f}")
print("smallest k with 95% energy:", kmin, "  rho(k) =", round(float(energy[kmin - 1]), 4))
assert kmin == 5
assert abs(float(energy[4]) - 0.9688) < 5e-5

k=1  retained energy 0.5000   relative error 0.7071
k=2  retained energy 0.7500   relative error 0.5000
k=3  retained energy 0.8750   relative error 0.3536
k=4  retained energy 0.9375   relative error 0.2500
k=5  retained energy 0.9688   relative error 0.1768
k=6  retained energy 0.9844   relative error 0.1250
k=7  retained energy 0.9922   relative error 0.0884
smallest k with 95% energy: 5   rho(k) = 0.9688


### Problem L2.5 — Singular values of the inverse and the cost of conditioning

**Statement.** Let $A \in \mathbb{R}^{n \times n}$ be invertible with singular values
$\sigma_1 \ge \dots \ge \sigma_n \gt 0$. Show $\sigma_i(A^{-1}) = 1/\sigma_{n-i+1}(A)$, and deduce
$\kappa_2(A) = \lVert A \rVert_{\mathrm{op}}\lVert A^{-1} \rVert_{\mathrm{op}}$.

**Intuition.** Inverting a map inverts every stretch factor, which reverses their order.

**Solution.**

*Step 1.* From $A = U\Sigma V^{\top}$ with all $\sigma_i \gt 0$,

$$
A^{-1} = (U\Sigma V^{\top})^{-1} = V\Sigma^{-1}U^{\top},
\qquad \Sigma^{-1} = \operatorname{diag}(1/\sigma_1, \dots, 1/\sigma_n) .
$$

*Step 2.* The diagonal of $\Sigma^{-1}$ ascends, so it is not yet in the order Definition 3.4
requires. Let $P$ be the reversal permutation matrix; then

$$
A^{-1} = (VP)(P\Sigma^{-1}P)(UP)^{\top},
$$

with $VP$ and $UP$ orthogonal and $P\Sigma^{-1}P = \operatorname{diag}(1/\sigma_n, \dots, 1/\sigma_1)$
descending.

*Step 3.* Hence $\sigma_1(A^{-1}) = 1/\sigma_n$ and $\lVert A^{-1} \rVert_{\mathrm{op}} = 1/\sigma_n$,
so $\lVert A \rVert_{\mathrm{op}}\lVert A^{-1} \rVert_{\mathrm{op}} = \sigma_1/\sigma_n$.

$$
\boxed{\sigma_i(A^{-1}) = \frac{1}{\sigma_{n-i+1}(A)}, \qquad \kappa_2(A) = \lVert A \rVert_{\mathrm{op}} \lVert A^{-1} \rVert_{\mathrm{op}}}
$$

**Key takeaway.** The *smallest* singular value controls how badly a linear solve can amplify
error, which is why $\sigma_n$ and not $\sigma_1$ is the quantity to watch.

In [29]:
A = rng.standard_normal((5, 5))
sA = np.linalg.svd(A, compute_uv=False)
sAinv = np.linalg.svd(np.linalg.inv(A), compute_uv=False)
print("sigma(A)      :", sA)
print("sigma(A^-1)   :", sAinv)
print("1/sigma(A) rev:", (1.0 / sA)[::-1])
print("kappa_2(A)    :", np.linalg.cond(A), " = ||A|| ||A^-1|| =",
      np.linalg.norm(A, 2) * np.linalg.norm(np.linalg.inv(A), 2))
assert np.allclose(sAinv, (1.0 / sA)[::-1])
assert np.isclose(np.linalg.cond(A), np.linalg.norm(A, 2) * np.linalg.norm(np.linalg.inv(A), 2))

sigma(A)      : [4.6505 3.0506 1.3019 0.9241 0.4439]
sigma(A^-1)   : [2.2525 1.0822 0.7681 0.3278 0.215 ]
1/sigma(A) rev: [2.2525 1.0822 0.7681 0.3278 0.215 ]
kappa_2(A)    : 10.475396305261397  = ||A|| ||A^-1|| = 10.47539630526139


### Problem L2.6 — CUR: the optimal core matrix

**Statement.** Fix a column selection $C \in \mathbb{R}^{m \times c}$ and a row selection
$R \in \mathbb{R}^{r \times n}$ taken from $A$. Prove that the core minimizing
$\lVert A - CMR \rVert_F$ over $M \in \mathbb{R}^{c \times r}$ is $M = C^{+}AR^{+}$.

**Intuition.** As $M$ ranges over everything, $CMR$ sweeps out a linear subspace of matrices, and
the closest point of a subspace is the orthogonal projection.

**Solution.**

*Step 1 — identify the search set.* Let
$\mathcal{S} = \lbrace CMR : M \in \mathbb{R}^{c \times r} \rbrace$. Every element has column space
inside $\operatorname{Col}(C)$ and row space inside $\operatorname{Row}(R)$, and conversely any such
matrix is $CMR$ for some $M$. $\mathcal{S}$ is a linear subspace of $\mathbb{R}^{m \times n}$.

*Step 2 — the projector onto it.* Put $P_C = CC^{+}$ and $P_R = R^{+}R$, the orthogonal projectors
onto $\operatorname{Col}(C)$ and $\operatorname{Row}(R)$ supplied by Theorem 4.7. Define
$\Pi(Z) = P_C Z P_R$.

$\Pi$ maps into $\mathcal{S}$, fixes $\mathcal{S}$ pointwise, and is self-adjoint for the Frobenius
inner product $\langle X, Y\rangle = \operatorname{tr}(X^{\top}Y)$:

$$
\langle P_C X P_R, \ Y \rangle = \operatorname{tr}(P_R X^{\top}P_C Y) = \operatorname{tr}(X^{\top}P_C Y P_R) = \langle X, \ P_C Y P_R \rangle,
$$

using $P_C^{\top} = P_C$, $P_R^{\top} = P_R$ and cyclicity. So $\Pi$ is *the* orthogonal projector
onto $\mathcal{S}$.

*Step 3 — read off the minimizer.* The closest element of $\mathcal{S}$ to $A$ is
$\Pi(A) = P_C A P_R = C(C^{+}AR^{+})R$, and the corresponding core is $M = C^{+}AR^{+}$.

$$
\boxed{M^{\star} = C^{+}AR^{+}, \qquad \min_M \lVert A - CMR \rVert_F = \lVert A - CC^{+}AR^{+}R \rVert_F}
$$

**Key takeaway.** CUR keeps actual columns and rows of $A$, so it inherits sparsity and stays
interpretable, but by Theorem 4.6 it can never beat the truncated SVD at the same rank. The
relative-error guarantee $(1+\epsilon)\lVert A - A_k\rVert_F$ for leverage-score sampling is
Drineas, Mahoney and Muthukrishnan, *SIAM J. Matrix Anal. Appl.* **30**(2), 2008, Theorem 4.

In [30]:
Acur = rng.standard_normal((14, 5)) @ rng.standard_normal((5, 11)) + 0.05 * rng.standard_normal((14, 11))
cols, rows = [0, 3, 5, 7, 9], [1, 2, 6, 9, 11, 13]
Cm = Acur[:, cols]
Rm = Acur[rows, :]
Mstar = np.linalg.pinv(Cm) @ Acur @ np.linalg.pinv(Rm)
best = np.linalg.norm(Acur - Cm @ Mstar @ Rm)
PCm = Cm @ np.linalg.pinv(Cm)
PRm = np.linalg.pinv(Rm) @ Rm
print("||A - C M* R||_F         :", best)
print("||C M* R - P_C A P_R||_F :", np.linalg.norm(Cm @ Mstar @ Rm - PCm @ Acur @ PRm))
worst = min(np.linalg.norm(Acur - Cm @ (Mstar + 0.05 * rng.standard_normal(Mstar.shape)) @ Rm)
            for _ in range(300))
print("best over 300 perturbed cores:", worst, " (>= optimum:", worst >= best - 1e-12, ")")
k_ref = min(len(cols), len(rows))
s_ref = np.linalg.svd(Acur, compute_uv=False)
print("truncated SVD at rank", k_ref, ":", np.sqrt((s_ref[k_ref:] ** 2).sum()), "<= CUR error")
assert np.linalg.norm(Cm @ Mstar @ Rm - PCm @ Acur @ PRm) < 1e-10
assert worst >= best - 1e-12
assert np.sqrt((s_ref[k_ref:] ** 2).sum()) <= best + 1e-12

||A - C M* R||_F         : 0.7531704368801316
||C M* R - P_C A P_R||_F : 7.507631314185799e-14
best over 300 perturbed cores: 6.6210896105513  (>= optimum: True )
truncated SVD at rank 5 : 0.33925269127238816 <= CUR error


### Problem L2.7 — The HOSVD core tensor is all-orthogonal

**Statement.** For a tensor $\mathcal{X} \in \mathbb{R}^{I_1 \times I_2 \times I_3}$, let
$X_{(n)}$ be its mode-$n$ unfolding, let $U^{(n)}$ come from the SVD
$X_{(n)} = U^{(n)}\Sigma_{(n)}W_{(n)}^{\top}$, and set

$$
\mathcal{G} = \mathcal{X} \times_1 (U^{(1)})^{\top} \times_2 (U^{(2)})^{\top} \times_3 (U^{(3)})^{\top} .
$$

Prove that for each mode the slices of $\mathcal{G}$ are mutually orthogonal, with
$\lVert \mathcal{G}_{i_n = \alpha} \rVert_F = \sigma_\alpha^{(n)}$.

**Intuition.** Unfolding turns each mode into an ordinary matrix problem, and the other two factor
matrices act as an orthogonal change of basis on the columns, which cannot change a Gram matrix.

**Solution.**

*Step 1 — unfold.* Mode-$n$ unfolding of a multilinear product gives

$$
G_{(n)} = (U^{(n)})^{\top} X_{(n)} \, W_n, \qquad
W_n = U^{(n+1)} \otimes U^{(n-1)} \ (\text{indices mod } 3),
$$

and $W_n$ is orthogonal because a Kronecker product of orthogonal matrices is orthogonal.

*Step 2 — substitute the SVD.* With $X_{(n)} = U^{(n)}\Sigma_{(n)}W_{(n)}^{\top}$ and
$(U^{(n)})^{\top}U^{(n)} = I$,

$$
G_{(n)} = \Sigma_{(n)} W_{(n)}^{\top} W_n .
$$

*Step 3 — form the Gram matrix of the slices.* The rows of $G_{(n)}$ are exactly the vectorized
mode-$n$ slices of $\mathcal{G}$, so their Gram matrix is

$$
G_{(n)}G_{(n)}^{\top} = \Sigma_{(n)} W_{(n)}^{\top} W_n W_n^{\top} W_{(n)} \Sigma_{(n)}^{\top} = \Sigma_{(n)}\Sigma_{(n)}^{\top},
$$

which is diagonal with entries $(\sigma_\alpha^{(n)})^2$.

$$
\boxed{\langle \mathcal{G}_{i_n = \alpha}, \ \mathcal{G}_{i_n = \beta}\rangle = 0 \ (\alpha \neq \beta), \qquad \lVert \mathcal{G}_{i_n = \alpha}\rVert_F = \sigma_\alpha^{(n)}}
$$

**Key takeaway.** All-orthogonality is the tensor stand-in for diagonality: the core cannot be made
diagonal in general, but its slices can always be made orthogonal with decreasing norms. Truncating
the HOSVD is therefore *quasi*-optimal, within $\sqrt{3}$ of the best Tucker approximation, not
optimal: the truncated HOSVD satisfies
$\lVert \mathcal{X} - \widehat{\mathcal{X}}\rVert_F \le \sqrt{3}\,\lVert \mathcal{X} - \mathcal{X}_{\text{best}}\rVert_F$
(De Lathauwer, De Moor and Vandewalle, *SIAM J. Matrix Anal. Appl.* **21**(4), 2000, Property 10),
and the direct analogue of Theorem 4.6 is false for tensors.

In [31]:
def unfold(T, mode):
    return np.moveaxis(T, mode, 0).reshape(T.shape[mode], -1)


Xten = rng.standard_normal((4, 3, 5))
factors = [np.linalg.svd(unfold(Xten, n), full_matrices=False)[0] for n in range(3)]
Gcore = np.einsum("ijk,ia,jb,kc->abc", Xten, factors[0], factors[1], factors[2])
for n in range(3):
    Gn = unfold(Gcore, n)
    Gram = Gn @ Gn.T
    sv_mode = np.linalg.svd(unfold(Xten, n), compute_uv=False)[:Gn.shape[0]]
    off = np.linalg.norm(Gram - np.diag(np.diag(Gram)))
    print(f"mode {n}: off-diagonal Gram mass {off:.2e}   slice norms",
          np.round(np.sqrt(np.diag(Gram)), 4), "  mode-n singular values", np.round(sv_mode, 4))
    assert off < 1e-11
    assert np.allclose(np.sqrt(np.diag(Gram)), sv_mode)
print("||X||_F, ||G||_F:", np.linalg.norm(Xten), np.linalg.norm(Gcore))
assert abs(np.linalg.norm(Xten) - np.linalg.norm(Gcore)) < 1e-12

mode 0: off-diagonal Gram mass 1.55e-14   slice norms [5.2541 3.915  2.5663 1.7501]   mode-n singular values [5.2541 3.915  2.5663 1.7501]
mode 1: off-diagonal Gram mass 4.28e-15   slice norms [5.4111 3.9805 2.7307]   mode-n singular values [5.4111 3.9805 2.7307]
mode 2: off-diagonal Gram mass 1.70e-14   slice norms [4.8746 3.3558 2.9138 2.3466 1.8871]   mode-n singular values [4.8746 3.3558 2.9138 2.3466 1.8871]
||X||_F, ||G||_F: 7.25127288321419 7.2512728832141935


### Problem L2.8 — Tangent space of the fixed-rank manifold

**Statement.** Let $\mathcal{M}_k = \lbrace A \in \mathbb{R}^{m \times n} : \operatorname{rank}(A) = k\rbrace$
and let $A = U\Sigma V^{\top}$ be a compact SVD, $U^{\top}U = V^{\top}V = I_k$. The tangent space at
$A$ is

$$
T_A\mathcal{M}_k = \lbrace UMV^{\top} + U_pV^{\top} + UV_p^{\top} \ : \ M \in \mathbb{R}^{k \times k}, \ U^{\top}U_p = 0, \ V^{\top}V_p = 0 \rbrace .
$$

Prove that the orthogonal projection of $Z \in \mathbb{R}^{m \times n}$ onto $T_A\mathcal{M}_k$ is

$$
\Pi_{T_A}(Z) = P_U Z + Z P_V - P_U Z P_V, \qquad P_U = UU^{\top}, \ P_V = VV^{\top},
$$

and compute $\dim T_A\mathcal{M}_k$.

**Intuition.** Split both the row and the column space into "inside the factors" and "outside".
The tangent space is everything except the outside-outside corner.

**Solution.**

*Step 1 — a four-block decomposition.* With $P_U^{\perp} = I_m - P_U$ and
$P_V^{\perp} = I_n - P_V$,

$$
Z = P_U Z P_V + P_U Z P_V^{\perp} + P_U^{\perp}Z P_V + P_U^{\perp}ZP_V^{\perp},
$$

and the four terms are mutually orthogonal in the Frobenius inner product, because
$\operatorname{tr}\bigl( (P_UXP_V)^{\top}P_U^{\perp}YP_{V}^{\perp}\bigr)$ contains the factor
$P_VP_V^{\perp} = 0$, and similarly for the other pairs.

*Step 2 — identify $T_A\mathcal{M}_k$ among the blocks.* The three generators match the first
three blocks exactly: $UMV^{\top}$ is an arbitrary matrix with column space in
$\operatorname{Col}(U)$ and row space in $\operatorname{Col}(V)$; $U_pV^{\top}$ with
$U^{\top}U_p = 0$ is an arbitrary $P_U^{\perp}\cdot P_V$ block; and $UV_p^{\top}$ is an arbitrary
$P_U \cdot P_V^{\perp}$ block. Hence

$$
T_A\mathcal{M}_k = \lbrace Z : P_U^{\perp}ZP_V^{\perp} = 0 \rbrace ,
$$

a linear subspace.

*Step 3 — project.* Removing the one forbidden block is the orthogonal projection:

$$
\Pi_{T_A}(Z) = Z - P_U^{\perp}ZP_V^{\perp} = Z - (I - P_U)Z(I - P_V) = P_UZ + ZP_V - P_UZP_V .
$$

*Step 4 — dimension.* The three surviving blocks have $k^2$, $(m-k)k$ and $(n-k)k$ free parameters.

$$
\boxed{\Pi_{T_A}(Z) = UU^{\top}Z + ZVV^{\top} - UU^{\top}ZVV^{\top}, \qquad \dim T_A\mathcal{M}_k = k(m + n - k)}
$$

**Key takeaway.** Riemannian optimization on $\mathcal{M}_k$ needs only this projection plus a
retraction, and both cost $O((m+n)k^2)$ — never an $mn$-sized object. It is the geometry behind
low-rank matrix completion and behind fixed-rank fine-tuning.

In [32]:
m_g, n_g, k_g = 9, 7, 3
Ug, _ = np.linalg.qr(rng.standard_normal((m_g, k_g)))
Vg, _ = np.linalg.qr(rng.standard_normal((n_g, k_g)))
PU, PV = Ug @ Ug.T, Vg @ Vg.T
Z = rng.standard_normal((m_g, n_g))
PTZ = PU @ Z + Z @ PV - PU @ Z @ PV

Mrand = rng.standard_normal((k_g, k_g))
Up = (np.eye(m_g) - PU) @ rng.standard_normal((m_g, k_g))
Vp = (np.eye(n_g) - PV) @ rng.standard_normal((n_g, k_g))
Wtan = Ug @ Mrand @ Vg.T + Up @ Vg.T + Ug @ Vp.T

print("forbidden block of Pi(Z)  :", np.linalg.norm((np.eye(m_g) - PU) @ PTZ @ (np.eye(n_g) - PV)))
print("idempotence ||Pi(Pi Z)-Pi Z||:",
      np.linalg.norm(PU @ PTZ + PTZ @ PV - PU @ PTZ @ PV - PTZ))
print("<Z - Pi(Z), W> for W in T :", np.trace((Z - PTZ).T @ Wtan))
print("dim k(m+n-k)              :", k_g * (m_g + n_g - k_g))
assert np.linalg.norm((np.eye(m_g) - PU) @ PTZ @ (np.eye(n_g) - PV)) < 1e-12
assert abs(np.trace((Z - PTZ).T @ Wtan)) < 1e-11

forbidden block of Pi(Z)  : 8.777439728313524e-16
idempotence ||Pi(Pi Z)-Pi Z||: 1.3032011253867433e-15
<Z - Pi(Z), W> for W in T : -1.1657341758564144e-15
dim k(m+n-k)              : 39


### Problem L2.9 — Physics: polar decomposition of a deformation gradient

**Statement.** A two-dimensional body deforms with constant deformation gradient
$F = \begin{pmatrix} 2 & 1 \\ 0 & 1\end{pmatrix}$. Compute the right stretch tensor
$U = (F^{\top}F)^{1/2}$, the rotation $R$, the principal stretches and the area ratio. Then show
that superposing a rigid rotation $Q$ on the deformed configuration leaves $U$ unchanged.

**Intuition.** Theorem 4.8 splits the deformation into "how the material is stretched" and "how it
is then turned"; a constitutive law may see the first but must not see the second.

**Solution.**

*Step 1 — the right Cauchy-Green tensor.*

$$
C = F^{\top}F = \begin{pmatrix} 4 & 2 \\ 2 & 2 \end{pmatrix},
\qquad \operatorname{tr} C = 6, \qquad \det C = 4 .
$$

*Step 2 — its square root.* Using
$M^{1/2} = (M + \sqrt{\det M}\,I)/\sqrt{\operatorname{tr}M + 2\sqrt{\det M}}$ for $M \succ 0$ in two
dimensions,

$$
U = \frac{1}{\sqrt{10}}\begin{pmatrix} 6 & 2 \\ 2 & 4 \end{pmatrix},
\qquad U^2 = \frac{1}{10}\begin{pmatrix} 40 & 20 \\ 20 & 20 \end{pmatrix} = C \ \checkmark
$$

*Step 3 — the rotation.* $\det U = 2$, so
$U^{-1} = \tfrac{1}{2\sqrt{10}}\left[\begin{smallmatrix}4 & -2 \\ -2 & 6\end{smallmatrix}\right]$ and

$$
R = FU^{-1} = \frac{1}{\sqrt{10}}\begin{pmatrix} 3 & 1 \\ -1 & 3\end{pmatrix},
\qquad R^{\top}R = I, \quad \det R = 1 ,
$$

a proper rotation by $\arctan(-1/3) \approx -18.43$ degrees.

*Step 4 — principal stretches.* The eigenvalues of $C$ are $3 \pm \sqrt5$, so the principal
stretches are

$$
\lambda_1 = \sqrt{3+\sqrt5} \approx 2.2882, \qquad \lambda_2 = \sqrt{3-\sqrt5} \approx 0.8740,
$$

and the area ratio is $\lambda_1\lambda_2 = \sqrt{\det C} = \det F = 2$.

*Step 5 — frame indifference.* Replace $F$ by $QF$ with $Q^{\top}Q = I$. Then
$(QF)^{\top}(QF) = F^{\top}Q^{\top}QF = F^{\top}F = C$, so $U$ is unchanged while $R \mapsto QR$.

$$
\boxed{F = RU, \quad U = \tfrac{1}{\sqrt{10}}\begin{pmatrix} 6 & 2 \\ 2 & 4\end{pmatrix}, \quad R = \tfrac{1}{\sqrt{10}}\begin{pmatrix} 3 & 1 \\ -1 & 3\end{pmatrix}, \quad \lambda = 2.2882,\ 0.8740}
$$

**Key takeaway.** The stretch is the physics and the rotation is the observer. Constitutive laws
are written in terms of $C = F^{\top}F$ or $U$ precisely so that a rigid rotation of the sample
cannot change the predicted stress.

In [33]:
F = np.array([[2.0, 1.0], [0.0, 1.0]])
Uh = np.array([[6.0, 2.0], [2.0, 4.0]]) / np.sqrt(10.0)
Rh = np.array([[3.0, 1.0], [-1.0, 3.0]]) / np.sqrt(10.0)
C = F.T @ F
print("C = F^T F           :\n", C)
print("||U^2 - C||_F       : %.3e" % np.linalg.norm(Uh @ Uh - C))
print("||R U - F||_F       : %.3e" % np.linalg.norm(Rh @ Uh - F))
print("R^T R - I           : %.3e" % np.linalg.norm(Rh.T @ Rh - np.eye(2)))
print("det R, rotation deg :", np.linalg.det(Rh), np.degrees(np.arctan2(-1.0, 3.0)))
print("principal stretches :", np.sqrt(np.linalg.eigvalsh(C))[::-1],
      "  svd(F) =", np.linalg.svd(F, compute_uv=False))
print("area ratio, det F   :", np.prod(np.sqrt(np.linalg.eigvalsh(C))), np.linalg.det(F))
theta = 1.1
Qsup = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
print("frame indifference  : ||(QF)^T(QF) - C||_F = %.3e"
      % np.linalg.norm((Qsup @ F).T @ (Qsup @ F) - C))
assert np.linalg.norm(Uh @ Uh - C) < 1e-13
assert np.linalg.norm(Rh @ Uh - F) < 1e-13
assert abs(np.linalg.det(Rh) - 1.0) < 1e-13
assert np.allclose(np.sqrt(np.linalg.eigvalsh(C))[::-1], np.linalg.svd(F, compute_uv=False))
assert np.linalg.norm((Qsup @ F).T @ (Qsup @ F) - C) < 1e-13

C = F^T F           :
 [[4. 2.]
 [2. 2.]]
||U^2 - C||_F       : 4.441e-16
||R U - F||_F       : 2.227e-16
R^T R - I           : 1.117e-16
det R, rotation deg : 1.0 -18.43494882292201
principal stretches : [2.2882 0.874 ]   svd(F) = [2.2882 0.874 ]
area ratio, det F   : 2.0 2.0
frame indifference  : ||(QF)^T(QF) - C||_F = 0.000e+00


### Problem L2.10 — Physics: optimal rigid alignment of two point sets

**Statement.** Let $p_1, \dots, p_N$ and $q_1, \dots, q_N$ be two centred point sets in
$\mathbb{R}^n$. Find the rotation $R \in SO(n)$ minimizing $\sum_i \lVert q_i - Rp_i \rVert^2$, and
show that the determinant correction is needed.

**Intuition.** Expanding the squares turns the problem into maximizing a trace, which Theorem 4.8
solves — except that a reflection is orthogonal but not a rigid motion.

**Solution.**

*Step 1 — reduce to a trace.* Since $\lVert Rp_i \rVert = \lVert p_i \rVert$,

$$
\sum_i \lVert q_i - Rp_i \rVert^2 = \sum_i \lVert q_i \rVert^2 + \sum_i \lVert p_i \rVert^2 - 2 \operatorname{tr}(RH), \qquad H = \sum_i p_iq_i^{\top} ,
$$

so minimizing the left side is maximizing $\operatorname{tr}(RH)$.

*Step 2 — rotate into singular coordinates.* Let $H = U\Sigma V^{\top}$ and $Z = V^{\top}RU$, which
is orthogonal with $\det Z = \det(R)\det(U)\det(V) = \det(U)\det(V)$. Then

$$
\operatorname{tr}(RH) = \operatorname{tr}(RU\Sigma V^{\top}) = \operatorname{tr}(V^{\top}RU\Sigma) = \operatorname{tr}(Z\Sigma) = \sum_i \sigma_i Z_{ii} .
$$

*Step 3 — the unconstrained bound.* $\lvert Z_{ii}\rvert \le 1$ gives
$\operatorname{tr}(Z\Sigma) \le \sum_i \sigma_i$, attained at $Z = I$. If
$\det(U)\det(V) = 1$ that choice is admissible and $R = VU^{\top}$.

*Step 4 — the constrained bound when $\det(U)\det(V) = -1$.* Write
$\Sigma = \sum_{j=1}^{n}(\sigma_j - \sigma_{j+1})P_j$ with $P_j$ the projector onto the first $j$
coordinates and $\sigma_{n+1} = 0$; every coefficient is non-negative. Then

$$
\operatorname{tr}(Z\Sigma) = \sum_{j=1}^{n}(\sigma_j - \sigma_{j+1}) \sum_{i \le j} Z_{ii} .
$$

For $j \lt n$ use $\sum_{i \le j}Z_{ii} \le j$. For $j = n$ use
$\operatorname{tr}Z \le n - 2$, which holds because an orthogonal $Z$ with $\det Z = -1$ has at
least one eigenvalue $-1$ and its remaining eigenvalues contribute at most $n-1$. Summing,

$$
\operatorname{tr}(Z\Sigma) \le \sum_{j \lt n} j(\sigma_j - \sigma_{j+1}) + (n-2)\sigma_n = \sigma_1 + \dots + \sigma_{n-1} - \sigma_n ,
$$

attained at $Z = \operatorname{diag}(1,\dots,1,-1)$.

*Step 5 — assemble.* Both cases are covered by inserting $d = \det(VU^{\top})$:

$$
\boxed{R^{\star} = V \operatorname{diag}(1, \dots, 1, d)\, U^{\top}, \qquad d = \det(VU^{\top})}
$$

**Key takeaway.** Without the correction the "optimal" transform can be a reflection, which fits
mirror-image data better than any rotation can. The code cell below builds exactly that case: the
unconstrained fit is a reflection with sum of squares $0.0058$, while the best proper rotation is
stuck at $0.503$.

In [34]:
theta = 0.5
Rtrue = np.array([[np.cos(theta), -np.sin(theta), 0.0],
                  [np.sin(theta), np.cos(theta), 0.0],
                  [0.0, 0.0, 1.0]])
Pset = rng.standard_normal((8, 3))
Pset -= Pset.mean(axis=0)


def kabsch(P, Q):
    H = P.T @ Q
    U, s, Vt = np.linalg.svd(H)
    naive = Vt.T @ U.T
    d = np.sign(np.linalg.det(naive))
    return Vt.T @ np.diag([1.0, 1.0, d]) @ U.T, naive, s


Qrot = Pset @ Rtrue.T + 0.02 * rng.standard_normal((8, 3))
Qrot -= Qrot.mean(axis=0)
Rfit, Rnaive, sing = kabsch(Pset, Qrot)
print("rotated data:  det(naive) = %+.6f   ||R_fit - R_true||_F = %.6f"
      % (np.linalg.det(Rnaive), np.linalg.norm(Rfit - Rtrue)))
assert np.linalg.det(Rnaive) > 0 and np.linalg.norm(Rfit - Rtrue) < 0.05

Qref = Pset @ np.diag([1.0, 1.0, -1.0]) + 0.02 * rng.standard_normal((8, 3))
Qref -= Qref.mean(axis=0)
Rfit2, Rnaive2, sing2 = kabsch(Pset, Qref)
sse_naive = ((Qref - Pset @ Rnaive2.T) ** 2).sum()
sse_proper = ((Qref - Pset @ Rfit2.T) ** 2).sum()
print("reflected data: det(naive) = %+.6f   det(corrected) = %+.6f"
      % (np.linalg.det(Rnaive2), np.linalg.det(Rfit2)))
print("  sum of squares, reflection = %.6f   best proper rotation = %.6f"
      % (sse_naive, sse_proper))
print("  trace bound: sum sigma = %.6f   sigma_1+sigma_2-sigma_3 = %.6f"
      % (sing2.sum(), sing2[0] + sing2[1] - sing2[2]))
assert np.linalg.det(Rnaive2) < 0 < np.linalg.det(Rfit2)
assert sse_proper > sse_naive

rotated data:  det(naive) = +1.000000   ||R_fit - R_true||_F = 0.025447
reflected data: det(naive) = -1.000000   det(corrected) = +1.000000
  sum of squares, reflection = 0.005826   best proper rotation = 0.502803
  trace bound: sum sigma = 13.886319   sigma_1+sigma_2-sigma_3 = 13.637830


## L3 — Challenge Proofs

### Problem L3.1 — Uniqueness of the Moore-Penrose pseudoinverse

**Statement.** Prove that a matrix $A$ has at most one $X$ satisfying all four Penrose conditions
of Definition 3.6.

**Intuition.** The four conditions are exactly enough to let one rewrite $X$ as $XAY$ and $Y$ as
$XAY$ for any second candidate $Y$.

**Solution.** Let $X$ and $Y$ both satisfy (P1)-(P4).

*Step 1 — push $X$ towards $XAY$.* Using (P2), then (P3) for $X$, then (P1) for $Y$:

$$
X = XAX = X(AX)^{\top} = XX^{\top}A^{\top} = XX^{\top}(AYA)^{\top} = XX^{\top}A^{\top}Y^{\top}A^{\top} .
$$

*Step 2 — fold the transposes back.* The last expression is $X(AX)^{\top}(AY)^{\top}$, and (P3)
holds for both $X$ and $Y$, so it equals $X(AX)(AY) = (XAX)AY = XAY$.

*Step 3 — the mirror computation.* Using (P2), then (P4) for $Y$, then (P1) for $X$:

$$
Y = YAY = (YA)^{\top}Y = A^{\top}Y^{\top}Y = (AXA)^{\top}Y^{\top}Y = A^{\top}X^{\top}A^{\top}Y^{\top}Y = (XA)^{\top}(YA)^{\top}Y ,
$$

which by (P4) for both equals $(XA)(YA)Y = XA(YAY) = XAY$.

*Step 4.* Both equal $XAY$.

$$
\boxed{X = XAY = Y, \qquad \text{so } A^{+} \text{ is unique}}
$$

**Key takeaway.** Existence is a construction from the SVD (Proof 5.8); uniqueness is pure algebra
and needs no factorization at all. Dropping either symmetry condition destroys it: without (P3) and
(P4) there are infinitely many generalized inverses.

In [35]:
A = rng.standard_normal((6, 3)) @ rng.standard_normal((3, 5))
X = np.linalg.pinv(A)
conds = [np.linalg.norm(A @ X @ A - A), np.linalg.norm(X @ A @ X - X),
         np.linalg.norm((A @ X).T - A @ X), np.linalg.norm((X @ A).T - X @ A)]
print("Penrose residuals for pinv(A):", ["%.2e" % c for c in conds])
Ug, sg, Vtg = np.linalg.svd(A, full_matrices=False)
tol = max(A.shape) * EPS * sg[0]
Y = Vtg.T @ np.diag([1 / x if x > tol else 0.0 for x in sg]) @ Ug.T
print("second construction differs by :", np.linalg.norm(X - Y))
print("a generalized inverse satisfying (P1) but not (P3):")
G = X + rng.standard_normal((5, 6)) @ (np.eye(6) - A @ X)
print("  A G A - A      :", "%.2e" % np.linalg.norm(A @ G @ A - A))
print("  (A G)^T - A G  :", "%.2e" % np.linalg.norm((A @ G).T - A @ G))
assert max(conds) < 1e-12
assert np.linalg.norm(X - Y) < 1e-12
assert np.linalg.norm(A @ G @ A - A) < 1e-10 < np.linalg.norm((A @ G).T - A @ G)

Penrose residuals for pinv(A): ['1.92e-15', '6.26e-16', '1.01e-15', '1.62e-15']
second construction differs by : 9.982293620279108e-17
a generalized inverse satisfying (P1) but not (P3):
  A G A - A      : 1.06e-14
  (A G)^T - A G  : 1.64e+01


### Problem L3.2 — Weyl's inequality for singular values from Courant-Fischer

**Statement.** Prove $\lvert \sigma_i(A+B) - \sigma_i(A)\rvert \le \lVert B \rVert_{\mathrm{op}}$
for all $A, B \in \mathbb{R}^{m \times n}$ and all $i$, using the variational characterization of
singular values rather than the route of Proof 5.6.

**Intuition.** Each singular value is a max-min of $\lVert Ax \rVert$; a pointwise perturbation of
the objective by at most $\lVert B \rVert_{\mathrm{op}}$ moves any max-min by at most that much.

**Solution.**

*Step 1 — Courant-Fischer for singular values.* Since
$\lVert Ax \rVert^2 = x^{\top}A^{\top}Ax$, applying the min-max theorem for symmetric matrices
([Module 06](../06_eigenvalues_eigenvectors_spectral_theory/first_principles.ipynb), Theorem 4.4)
to $A^{\top}A$ gives

$$
\sigma_i(A)^2 = \lambda_i(A^{\top}A) = \max_{\dim S = i} \ \min_{x \in S, \lVert x \rVert = 1} \lVert Ax \rVert^2 ,
$$

and taking square roots, which is monotone,

$$
\sigma_i(A) = \max_{\dim S = i} \ \min_{x \in S, \lVert x \rVert = 1} \lVert Ax \rVert .
$$

*Step 2 — a pointwise bound.* For every unit $x$,
$\lVert (A+B)x \rVert \le \lVert Ax \rVert + \lVert Bx \rVert \le \lVert Ax \rVert + \lVert B \rVert_{\mathrm{op}}$.

*Step 3 — pass to the minimum over a fixed subspace.* Fix $S$ of dimension $i$ and let $x_S$
attain $\min_{x \in S, \lVert x\rVert = 1}\lVert Ax \rVert$. Then

$$
\min_{x \in S, \lVert x \rVert = 1} \lVert (A+B)x \rVert \ \le\ \lVert (A+B)x_S \rVert \ \le\ \lVert Ax_S \rVert + \lVert B \rVert_{\mathrm{op}} = \min_{x \in S, \lVert x\rVert = 1}\lVert Ax \rVert + \lVert B \rVert_{\mathrm{op}} .
$$

*Step 4 — pass to the maximum over $S$.* Taking $\max_{\dim S = i}$ of both sides gives
$\sigma_i(A+B) \le \sigma_i(A) + \lVert B \rVert_{\mathrm{op}}$.

*Step 5 — the other direction.* Apply Step 4 with $A + B$ in place of $A$ and $-B$ in place of
$B$, using $\lVert -B\rVert_{\mathrm{op}} = \lVert B \rVert_{\mathrm{op}}$.

$$
\boxed{\lvert \sigma_i(A+B) - \sigma_i(A) \rvert \ \le\ \lVert B \rVert_{\mathrm{op}}}
$$

**Key takeaway.** Singular values are $1$-Lipschitz in the operator norm with no condition-number
factor, unlike eigenvalues of a non-normal matrix. This is Theorem 4.5 with $j = 1$, obtained here
without the low-rank machinery.

In [36]:
A = rng.standard_normal((7, 5))
for scale in (1.0, 0.1, 0.01):
    B = scale * rng.standard_normal((7, 5))
    sA, sAB = np.linalg.svd(A, compute_uv=False), np.linalg.svd(A + B, compute_uv=False)
    gap = np.abs(sAB - sA).max()
    print(f"||B||_op = {np.linalg.norm(B, 2):.6f}   max_i |sigma_i(A+B)-sigma_i(A)| = {gap:.6f}"
          f"   satisfied: {gap <= np.linalg.norm(B, 2) + 1e-12}")
    assert gap <= np.linalg.norm(B, 2) + 1e-12
Ua, sa, Vta = np.linalg.svd(A, full_matrices=False)
tsh = 0.5
Bsharp = -tsh * np.outer(Ua[:, 0], Vta[0])
s_sharp = np.linalg.svd(A + Bsharp, compute_uv=False)
print("sharpness: B = -0.5 u_1 v_1^T")
print(f"  ||B||_op = {np.linalg.norm(Bsharp, 2):.6f}"
      f"   sigma_1 shift = {abs(s_sharp[0] - sa[0]):.6f}"
      f"   other shifts = {np.abs(s_sharp[1:] - sa[1:]).max():.2e}")
assert abs(abs(s_sharp[0] - sa[0]) - tsh) < 1e-12

||B||_op = 4.029366   max_i |sigma_i(A+B)-sigma_i(A)| = 2.073839   satisfied: True
||B||_op = 0.461700   max_i |sigma_i(A+B)-sigma_i(A)| = 0.130390   satisfied: True
||B||_op = 0.044074   max_i |sigma_i(A+B)-sigma_i(A)| = 0.022731   satisfied: True
sharpness: B = -0.5 u_1 v_1^T
  ||B||_op = 0.500000   sigma_1 shift = 0.500000   other shifts = 1.78e-15


### Problem L3.3 — Non-uniqueness of the SVD under a repeated singular value

**Statement.** Suppose $\sigma_1 = \sigma_2 = \sigma \gt 0$. Show that $u_1, u_2$ and $v_1, v_2$
are determined only up to a common $2 \times 2$ orthogonal mixing, and that this is the whole
freedom.

**Intuition.** On the repeated block $\Sigma$ acts as $\sigma I_2$, which commutes with every
orthogonal matrix, so a rotation applied to both sides cancels.

**Solution.**

*Step 1 — the mixing works.* Let $R \in \mathbb{R}^{2\times2}$ be orthogonal and set
$[\tilde{u}_1\ \tilde{u}_2] = [u_1\ u_2]R$, $[\tilde{v}_1\ \tilde{v}_2] = [v_1\ v_2]R$. Both remain
orthonormal because $R$ is orthogonal. The contribution of the block to $A$ is unchanged:

$$
[\tilde{u}_1\ \tilde{u}_2](\sigma I_2)[\tilde{v}_1\ \tilde{v}_2]^{\top}
= \sigma [u_1\ u_2] RR^{\top}[v_1\ v_2]^{\top}
= [u_1\ u_2](\sigma I_2)[v_1\ v_2]^{\top} .
$$

*Step 2 — nothing else works.* By Theorem 4.4 statement 3, $v_1$ and $v_2$ must span the
eigenspace of $A^{\top}A$ for $\sigma^2$, so any admissible replacement is $[v_1\ v_2]R$ with $R$
orthogonal. Once $V$ is fixed, $u_i = Av_i/\sigma$ is forced, and

$$
[\tilde{u}_1\ \tilde{u}_2] = \frac{1}{\sigma}A[v_1\ v_2]R = [u_1\ u_2]R ,
$$

so the same $R$ must be used on both sides.

*Step 3 — the distinct case.* If $\sigma_i$ is simple and positive, the eigenspace is
one-dimensional, $R = [\pm 1]$, and the freedom collapses to a sign shared by $u_i$ and $v_i$.

$$
\boxed{\text{singular vectors are unique up to a shared orthogonal mixing within each repeated } \sigma}
$$

**Key takeaway.** Singular *values* are always unique; singular *vectors* are not. Any algorithm
whose output is a subspace — PCA, LSA, spectral clustering — must be stated in terms of the span,
never the individual vectors.

In [37]:
Rot, _ = np.linalg.qr(rng.standard_normal((2, 2)))
Uf, _ = np.linalg.qr(rng.standard_normal((5, 5)))
Vf, _ = np.linalg.qr(rng.standard_normal((4, 4)))
Sg = np.zeros((5, 4))
np.fill_diagonal(Sg, [3.0, 3.0, 1.5, 0.5])
A = Uf @ Sg @ Vf.T

Um = Uf.copy(); Vm = Vf.copy()
Um[:, :2] = Uf[:, :2] @ Rot
Vm[:, :2] = Vf[:, :2] @ Rot
print("||U' S V'^T - A||_F      :", np.linalg.norm(Um @ Sg @ Vm.T - A))
print("angle between v1 and v1' :",
      np.degrees(np.arccos(abs(Vf[:, 0] @ Vm[:, 0]))), "degrees")
Vwrong = Vf.copy(); Vwrong[:, :2] = Vf[:, :2] @ Rot
print("mixing only V (not U)    :", np.linalg.norm(Uf @ Sg @ Vwrong.T - A))
print("singular values unchanged:", np.linalg.svd(Um @ Sg @ Vm.T, compute_uv=False))
assert np.linalg.norm(Um @ Sg @ Vm.T - A) < 1e-13
assert np.linalg.norm(Uf @ Sg @ Vwrong.T - A) > 0.1

||U' S V'^T - A||_F      : 1.6148402845793503e-15
angle between v1 and v1' : 30.043000736565276 degrees
mixing only V (not U)    : 5.999999999999999
singular values unchanged: [3.  3.  1.5 0.5]


### Problem L3.4 — Characteristic and minimal polynomial from the Jordan form

**Statement.** Let $J = \operatorname{diag}\bigl(J_3(\lambda), J_2(\lambda), J_1(\mu)\bigr)$ with
$\lambda \neq \mu$. Find $p_J$ and $m_J$, and prove the general rule.

**Intuition.** A polynomial annihilates a block diagonal matrix exactly when it annihilates every
block, and for one Jordan block the condition is that the polynomial vanish to the block's order.

**Solution.**

*Step 1 — one block.* Write $J_k(\nu) = \nu I + N_k$ with $N_k^k = 0$, $N_k^{k-1} \neq 0$. Taylor
expansion of a polynomial $q$ around $\nu$ gives

$$
q\bigl( J_k(\nu) \bigr) = \sum_{j=0}^{k-1} \frac{q^{(j)}(\nu)}{j!} N_k^{\,j} .
$$

The matrices $I, N_k, \dots, N_k^{k-1}$ are linearly independent, so
$q(J_k(\nu)) = 0$ if and only if $q(\nu) = q'(\nu) = \dots = q^{(k-1)}(\nu) = 0$, that is if and
only if $(x - \nu)^k$ divides $q$.

*Step 2 — assemble.* $q(J) = 0$ iff $q$ annihilates every block, so the monic generator of the
annihilating ideal is

$$
m_J(x) = \prod_{\nu} (x - \nu)^{k_{\max}(\nu)} ,
$$

with $k_{\max}(\nu)$ the largest block size for $\nu$.

*Step 3 — the characteristic polynomial.* $J$ is triangular, so
$p_J(x) = \prod_\nu (x-\nu)^{m_\nu}$ with $m_\nu$ the total of the block sizes.

*Step 4 — the given case.* For $\lambda$ the sizes are $3$ and $2$, so $m_\lambda = 5$ and
$k_{\max} = 3$; for $\mu$ they are just $1$.

$$
\boxed{p_J(x) = (x-\lambda)^5(x-\mu), \qquad m_J(x) = (x-\lambda)^3(x-\mu)}
$$

**Key takeaway.** The minimal polynomial records only the largest block per eigenvalue, so it
cannot distinguish $J_3 \oplus J_2$ from $J_3 \oplus J_1$; the full block structure needs the rank
sequence of Theorem 4.2.

In [38]:
def jordan_block(lmbda, k):
    return lmbda * np.eye(k) + np.diag(np.ones(k - 1), 1)


lam, mu = 2.0, -1.0
Jbig = np.zeros((6, 6))
Jbig[:3, :3] = jordan_block(lam, 3)
Jbig[3:5, 3:5] = jordan_block(lam, 2)
Jbig[5, 5] = mu
print("characteristic polynomial roots:", np.sort(np.linalg.eigvals(Jbig).real))
for e in (1, 2, 3, 4):
    M = np.linalg.matrix_power(Jbig - lam * np.eye(6), e) @ (Jbig - mu * np.eye(6))
    print(f"  ||(J - lambda I)^{e} (J - mu I)||_F = {np.linalg.norm(M):.3e}")
alt = np.zeros((6, 6))
alt[:3, :3] = jordan_block(lam, 3)
alt[3, 3] = lam
alt[4, 4] = lam
alt[5, 5] = mu
same_min = np.linalg.norm(np.linalg.matrix_power(alt - lam * np.eye(6), 3)
                          @ (alt - mu * np.eye(6)))
print("a different Jordan form with the same minimal polynomial: residual %.3e" % same_min)
print("  rank sequences differ:",
      [int(np.linalg.matrix_rank(np.linalg.matrix_power(Jbig - lam * np.eye(6), j))) for j in range(4)],
      "vs",
      [int(np.linalg.matrix_rank(np.linalg.matrix_power(alt - lam * np.eye(6), j))) for j in range(4)])
assert np.linalg.norm(np.linalg.matrix_power(Jbig - lam * np.eye(6), 3)
                      @ (Jbig - mu * np.eye(6))) < 1e-12
assert np.linalg.norm(np.linalg.matrix_power(Jbig - lam * np.eye(6), 2)
                      @ (Jbig - mu * np.eye(6))) > 0.5
assert same_min < 1e-12

characteristic polynomial roots: [-1.  2.  2.  2.  2.  2.]
  ||(J - lambda I)^1 (J - mu I)||_F = 5.292e+00
  ||(J - lambda I)^2 (J - mu I)||_F = 3.000e+00
  ||(J - lambda I)^3 (J - mu I)||_F = 0.000e+00
  ||(J - lambda I)^4 (J - mu I)||_F = 0.000e+00
a different Jordan form with the same minimal polynomial: residual 0.000e+00
  rank sequences differ: [6, 4, 2, 1] vs [6, 3, 2, 1]


### Problem L3.5 — Singular values of a real skew-symmetric matrix

**Statement.** Let $A \in \mathbb{R}^{n \times n}$ satisfy $A^{\top} = -A$. Prove that its non-zero
singular values occur in equal pairs $\sigma = \lvert \omega \rvert$, where $\pm i\omega$ are the
non-zero eigenvalues of $A$.

**Intuition.** $A^{\top}A = -A^2$, and squaring a purely imaginary eigenvalue and negating gives a
positive number that is shared by the eigenvalue and its conjugate.

**Solution.**

*Step 1 — the spectrum is imaginary.* Let $Ax = \nu x$ with $x \in \mathbb{C}^n$, $x \neq 0$.
Since $A$ is real, $A^{\ast} = A^{\top} = -A$, so

$$
\overline{x^{\ast}Ax} = x^{\ast}A^{\ast}x = -x^{\ast}Ax ,
$$

which makes the scalar $x^{\ast}Ax = \nu \lVert x \rVert^2$ purely imaginary; hence
$\nu = i\omega$ with $\omega \in \mathbb{R}$.

*Step 2 — eigenvalues pair up.* $A$ is real, so its characteristic polynomial has real
coefficients and non-real roots come in conjugate pairs: $i\omega$ and $-i\omega$ have equal
multiplicity.

*Step 3 — $A$ is normal.* $A^{\top}A = -A^2 = AA^{\top}$, so by
[Module 06](../06_eigenvalues_eigenvectors_spectral_theory/first_principles.ipynb) Theorem 4.3
there is a unitary $U$ with $A = UDU^{\ast}$, $D = \operatorname{diag}(i\omega_1, \dots)$.

*Step 4 — read off the singular values.*

$$
A^{\top}A = A^{\ast}A = UD^{\ast}DU^{\ast} = U\operatorname{diag}(\lvert \omega_j\rvert^2)U^{\ast},
$$

so $\sigma_j = \lvert \omega_j \rvert$, and by Step 2 each non-zero value occurs an even number of
times.

$$
\boxed{\sigma_{2k-1} = \sigma_{2k} = \lvert \omega_k \rvert, \qquad \operatorname{rank}(A) \text{ is even}}
$$

**Key takeaway.** A skew-symmetric matrix of odd order is always singular, since its non-zero
singular values pair up and cannot fill an odd dimension. Geometrically it is a direct sum of
planar rotations scaled by $\omega_k$, plus a kernel.

In [39]:
for n in (4, 5, 6):
    M = rng.standard_normal((n, n))
    A = (M - M.T) / 2
    s = np.linalg.svd(A, compute_uv=False)
    ev = np.linalg.eigvals(A)
    print(f"n = {n}: singular values", np.round(s, 6))
    print(f"        |imag eigenvalues| sorted", np.round(np.sort(np.abs(ev.imag))[::-1], 6),
          "  max |real part| = %.2e" % np.abs(ev.real).max(), "  rank =", np.linalg.matrix_rank(A))
    assert np.abs(ev.real).max() < 1e-12
    assert np.allclose(s, np.sort(np.abs(ev.imag))[::-1])
    assert np.linalg.matrix_rank(A) % 2 == 0

n = 4: singular values [1.6383 1.6383 0.77   0.77  ]
        |imag eigenvalues| sorted [1.6383 1.6383 0.77   0.77  ]   max |real part| = 1.11e-16   rank = 4
n = 5: singular values [1.2996 1.2996 0.571  0.571  0.    ]
        |imag eigenvalues| sorted [1.2996 1.2996 0.571  0.571  0.    ]   max |real part| = 3.21e-17   rank = 4
n = 6: singular values [2.2736 2.2736 1.4405 1.4405 0.679  0.679 ]
        |imag eigenvalues| sorted [2.2736 2.2736 1.4405 1.4405 0.679  0.679 ]   max |real part| = 1.67e-16   rank = 6


### Problem L3.6 — Orthogonal exactly when every singular value is one

**Statement.** Prove that a real square $A$ satisfies $A^{\top}A = I_n$ if and only if
$\sigma_1 = \dots = \sigma_n = 1$.

**Intuition.** Singular values are the stretch factors; an isometry stretches nothing.

**Solution.**

*Step 1 — forwards.* If $A^{\top}A = I$ then by Theorem 4.4 statement 3 every $\sigma_i^2$ is an
eigenvalue of $I$, hence $\sigma_i = 1$.

*Step 2 — backwards.* If all $\sigma_i = 1$ then $\Sigma = I_n$ and $A = UI_nV^{\top} = UV^{\top}$,
so

$$
A^{\top}A = VU^{\top}UV^{\top} = VV^{\top} = I_n .
$$

*Step 3 — the geometric reading.* $\lVert Ax \rVert^2 = x^{\top}A^{\top}Ax = \lVert x \rVert^2$ for
every $x$, so the image ellipsoid is the sphere itself.

$$
\boxed{A^{\top}A = I_n \iff \sigma_1 = \dots = \sigma_n = 1 \iff \kappa_2(A) = 1 \text{ and } \lVert A \rVert_{\mathrm{op}} = 1}
$$

**Key takeaway.** This is why $\kappa_2 = 1$ for orthogonal matrices, and why factorizations built
from them — QR, Householder, the SVD itself — do not amplify rounding error.

In [40]:
Qo, _ = np.linalg.qr(rng.standard_normal((5, 5)))
print("singular values of an orthogonal matrix:", np.linalg.svd(Qo, compute_uv=False))
print("kappa_2                                :", np.linalg.cond(Qo))
Uo, _ = np.linalg.qr(rng.standard_normal((5, 5)))
Vo, _ = np.linalg.qr(rng.standard_normal((5, 5)))
Aone = Uo @ Vo
print("U V^T is orthogonal: ||A^T A - I||_F   :", np.linalg.norm(Aone.T @ Aone - np.eye(5)))
assert np.allclose(np.linalg.svd(Qo, compute_uv=False), np.ones(5))
assert np.linalg.norm(Aone.T @ Aone - np.eye(5)) < 1e-13

singular values of an orthogonal matrix: [1. 1. 1. 1. 1.]
kappa_2                                : 1.0000000000000007
U V^T is orthogonal: ||A^T A - I||_F   : 1.6276544306297645e-15


### Problem L3.7 — Variational characterization of the nuclear norm

**Statement.** For square $A$, prove
$\lVert A \rVert_{\ast} = \max_{W^{\top}W = I} \operatorname{tr}(W^{\top}A)$.

**Intuition.** The trace pairing is maximized by aligning $W$ with the rotational part of $A$,
which is exactly the polar factor.

**Solution.**

*Step 1 — rotate.* With $A = U\Sigma V^{\top}$ and $W$ orthogonal,

$$
\operatorname{tr}(W^{\top}A) = \operatorname{tr}(W^{\top}U\Sigma V^{\top}) = \operatorname{tr}\bigl( (V^{\top}W^{\top}U)\Sigma \bigr) = \operatorname{tr}(Z\Sigma), \qquad Z = V^{\top}W^{\top}U .
$$

$Z$ is a product of orthogonal matrices, hence orthogonal.

*Step 2 — bound.* The columns of an orthogonal matrix are unit vectors, so
$\lvert Z_{ii} \rvert \le 1$ and

$$
\operatorname{tr}(Z\Sigma) = \sum_{i}\sigma_i Z_{ii} \ \le\ \sum_i \sigma_i = \lVert A \rVert_{\ast} .
$$

*Step 3 — attain it.* Take $W = UV^{\top}$, the polar factor of Theorem 4.8. Then
$Z = V^{\top}(VU^{\top})U = I$ and $\operatorname{tr}(Z\Sigma) = \sum_i \sigma_i$.

$$
\boxed{\lVert A \rVert_{\ast} = \max_{W^{\top}W = I}\operatorname{tr}(W^{\top}A), \ \text{ attained at } W = UV^{\top}}
$$

**Key takeaway.** The nuclear norm is the dual of the operator norm, and this maximum is its
support-function form. Because it is a maximum of linear functions it is convex, which is why
$\lVert \cdot \rVert_{\ast}$ is the standard convex surrogate for rank in matrix completion.

In [41]:
A = rng.standard_normal((5, 5))
U, s, Vt = np.linalg.svd(A)
Wopt = U @ Vt
print("nuclear norm ||A||_*       :", s.sum(), np.linalg.norm(A, "nuc"))
print("tr(W_opt^T A)              :", np.trace(Wopt.T @ A))
best_random = max(np.trace(np.linalg.qr(rng.standard_normal((5, 5)))[0].T @ A) for _ in range(500))
print("best of 500 random orthogonal W:", best_random)
assert abs(np.trace(Wopt.T @ A) - s.sum()) < 1e-12
assert best_random <= s.sum() + 1e-12

nuclear norm ||A||_*       : 9.687352308743419 9.687352308743423
tr(W_opt^T A)              : 9.687352308743424
best of 500 random orthogonal W: 5.579105779231869


### Problem L3.8 — Deterministic error bound for the randomized range finder

**Statement.** Let $A = U\Sigma V^{\top}$ and split
$U = [\,U_1\ U_2\,]$, $\Sigma = \operatorname{diag}(\Sigma_1, \Sigma_2)$, $V = [\,V_1\ V_2\,]$
with $\Sigma_1$ holding the $k$ largest singular values. Let $\Omega \in \mathbb{R}^{n \times \ell}$
be any test matrix, put $\Omega_1 = V_1^{\top}\Omega$ and $\Omega_2 = V_2^{\top}\Omega$, and assume
$\Omega_1$ has full row rank. With $Y = A\Omega$ and $Q$ an orthonormal basis of
$\operatorname{Col}(Y)$, prove

$$
\lVert A - QQ^{\top}A \rVert_F^2 \ \le\ \lVert \Sigma_2 \rVert_F^2 + \lVert \Sigma_2 \Omega_2 \Omega_1^{+} \rVert_F^2 ,
$$

then deduce the expectation bound of Theorem 4.9 for Gaussian $\Omega$.

**Intuition.** $QQ^{\top}A$ is at least as good as *any* approximation built from the columns of
$Y$, so it suffices to exhibit one good choice.

**Solution.**

*Step 1 — the projector is optimal within the range.* Let $P = QQ^{\top}$, the orthogonal projector
onto $\operatorname{Col}(Y)$. For any $X \in \mathbb{R}^{\ell \times n}$ the columns of $YX$ lie in
$\operatorname{Col}(Y)$, so $(I - P)YX = 0$ and therefore

$$
\lVert (I-P)A \rVert_F = \lVert (I-P)(A - YX) \rVert_F \ \le\ \lVert A - YX \rVert_F ,
$$

since an orthogonal projector has operator norm at most $1$.

*Step 2 — choose $X$.* Take $X = \Omega_1^{+}V_1^{\top}$. Expanding
$A\Omega = U_1\Sigma_1\Omega_1 + U_2\Sigma_2\Omega_2$ and using
$\Omega_1\Omega_1^{+} = I_k$, valid because $\Omega_1$ has full row rank,

$$
YX = U_1\Sigma_1 V_1^{\top} + U_2\Sigma_2\Omega_2\Omega_1^{+}V_1^{\top} .
$$

*Step 3 — subtract.* Since $A = U_1\Sigma_1V_1^{\top} + U_2\Sigma_2V_2^{\top}$,

$$
A - YX = U_2 \Sigma_2 \bigl( V_2^{\top} - \Omega_2\Omega_1^{+}V_1^{\top} \bigr) .
$$

*Step 4 — take norms.* $U_2$ has orthonormal columns, so it drops out. The two remaining terms are
Frobenius-orthogonal because $V_1^{\top}V_2 = 0$ kills the cross term:

$$
\lVert A - YX \rVert_F^2 = \lVert \Sigma_2 V_2^{\top}\rVert_F^2 + \lVert \Sigma_2\Omega_2\Omega_1^{+}V_1^{\top}\rVert_F^2 = \lVert \Sigma_2 \rVert_F^2 + \lVert \Sigma_2\Omega_2\Omega_1^{+}\rVert_F^2 .
$$

*Step 5 — average over Gaussian $\Omega$.* For $\Omega$ with i.i.d. standard normal entries,
$\Omega_1 = V_1^{\top}\Omega$ and $\Omega_2 = V_2^{\top}\Omega$ are independent standard Gaussian
matrices, because $V$ is orthogonal. Conditioning on $\Omega_1$ and using
$\mathbb{E}\lVert M G N\rVert_F^2 = \lVert M \rVert_F^2\lVert N \rVert_F^2$ for a standard Gaussian
$G$,

$$
\mathbb{E}\lVert \Sigma_2\Omega_2\Omega_1^{+}\rVert_F^2 = \lVert \Sigma_2 \rVert_F^2 \, \mathbb{E}\lVert \Omega_1^{+}\rVert_F^2 = \lVert \Sigma_2 \rVert_F^2 \cdot \frac{k}{p-1},
$$

with $\ell = k + p$, $p \ge 2$; the Gaussian moment $\mathbb{E}\lVert \Omega_1^{+}\rVert_F^2 = k/(p-1)$
is Halko, Martinsson and Tropp, *SIAM Review* **53**(2), 2011, Proposition 10.2. Jensen's
inequality then gives

$$
\mathbb{E}\lVert A - QQ^{\top}A\rVert_F \le \bigl( \mathbb{E}\lVert A - QQ^{\top}A\rVert_F^2 \bigr)^{1/2} \le \Bigl(1 + \frac{k}{p-1}\Bigr)^{1/2}\lVert \Sigma_2 \rVert_F .
$$

$$
\boxed{\mathbb{E}\lVert A - QQ^{\top}A \rVert_F \ \le\ \Bigl(1 + \frac{k}{p-1}\Bigr)^{1/2} \lVert A - A_k \rVert_F}
$$

**Key takeaway.** Steps 1 to 4 are deterministic and hold for every test matrix; randomness enters
only in Step 5, to control $\lVert \Omega_1^{+}\rVert_F$. Oversampling by $p = 5$ or $10$ already
brings the constant close to $1$, which is why the algorithm works in practice.

In [42]:
m_h, n_h, k_h, p_h = 40, 25, 6, 4
Qlh, _ = np.linalg.qr(rng.standard_normal((m_h, n_h)))
Qrh, _ = np.linalg.qr(rng.standard_normal((n_h, n_h)))
sv_h = 1.0 / (1.0 + np.arange(n_h))
Ah = Qlh @ np.diag(sv_h) @ Qrh.T
Uh, sh, Vth = np.linalg.svd(Ah, full_matrices=False)
Vh = Vth.T
S2 = np.diag(sh[k_h:])

print("deterministic bound, three draws")
for _ in range(3):
    Om = rng.standard_normal((n_h, k_h + p_h))
    Qy, _ = np.linalg.qr(Ah @ Om)
    lhs = np.linalg.norm(Ah - Qy @ (Qy.T @ Ah)) ** 2
    O1, O2 = Vh[:, :k_h].T @ Om, Vh[:, k_h:].T @ Om
    rhs = np.linalg.norm(S2) ** 2 + np.linalg.norm(S2 @ O2 @ np.linalg.pinv(O1)) ** 2
    print(f"  lhs = {lhs:.6f}   rhs = {rhs:.6f}   holds: {lhs <= rhs + 1e-10}")
    assert lhs <= rhs + 1e-10

moments = [np.linalg.norm(np.linalg.pinv(rng.standard_normal((k_h, k_h + p_h)))) ** 2
           for _ in range(4000)]
print(f"E ||Omega_1^+||_F^2 measured {np.mean(moments):.4f}   predicted k/(p-1) = {k_h / (p_h - 1):.4f}")
errs = []
for _ in range(300):
    Om = rng.standard_normal((n_h, k_h + p_h))
    Qy, _ = np.linalg.qr(Ah @ Om)
    errs.append(np.linalg.norm(Ah - Qy @ (Qy.T @ Ah)))
opt = np.sqrt((sv_h[k_h:] ** 2).sum())
print(f"E||A - QQ^T A||_F measured {np.mean(errs):.6f}"
      f"   bound {np.sqrt(1 + k_h / (p_h - 1)) * opt:.6f}")
assert abs(np.mean(moments) - k_h / (p_h - 1)) < 0.1
assert np.mean(errs) <= np.sqrt(1 + k_h / (p_h - 1)) * opt

deterministic bound, three draws
  lhs = 0.154978   rhs = 0.352451   holds: True
  lhs = 0.169537   rhs = 0.311154   holds: True
  lhs = 0.145802   rhs = 0.275322   holds: True


E ||Omega_1^+||_F^2 measured 1.9954   predicted k/(p-1) = 2.0000


E||A - QQ^T A||_F measured 0.386595   bound 0.585665


### Problem L3.9 — Interpolative decomposition from a maximum-volume column choice

**Statement.** Let $A \in \mathbb{R}^{m \times n}$ have SVD $A = U\Sigma V^{\top}$ and let $V_k$
hold the first $k$ right singular vectors. Choose an index set $J$ of size $k$ maximizing
$\lvert \det V_k(J,:)\rvert$ and set $X = \bigl( V_k V_k(J,:)^{-1} \bigr)^{\top} \in \mathbb{R}^{k \times n}$.
Prove that $X(:,J) = I_k$, that $\lvert X_{ij}\rvert \le 1$, and that

$$
\lVert A - A(:,J)X \rVert_{\mathrm{op}} \ \le\ \bigl( 1 + \sqrt{k(n-k+1)} \bigr)\, \sigma_{k+1} .
$$

**Intuition.** The maximum-volume submatrix makes the interpolation coefficients bounded by $1$;
after that the error is controlled by the part of $A$ the top-$k$ subspace already misses.

**Solution.** Write $W = V_k V_k(J,:)^{-1} \in \mathbb{R}^{n \times k}$, so $X = W^{\top}$. The
inverse exists because $V_k$ has rank $k$, so some $k \times k$ submatrix is non-singular and the
maximum of $\lvert \det \rvert$ is positive.

*Step 1 — the identity block.* $W(J,:) = V_k(J,:)V_k(J,:)^{-1} = I_k$, so $X(:,J) = I_k$.

*Step 2 — the entries are bounded by $1$.* Fix $i \notin J$ and $j \le k$, and let $J'$ be $J$ with
its $j$-th index replaced by $i$. Cramer's rule applied to
$W(i,:) = V_k(i,:)V_k(J,:)^{-1}$ gives

$$
W_{ij} = \frac{\det V_k(J',:)}{\det V_k(J,:)} ,
$$

and $\lvert \det V_k(J',:)\rvert \le \lvert \det V_k(J,:)\rvert$ by maximality, so
$\lvert W_{ij} \rvert \le 1$. Rows with $i \in J$ are rows of $I_k$, also bounded by $1$.

*Step 3 — the rank-$k$ part is reproduced exactly.* With $A_k = U_k\Sigma_kV_k^{\top}$,

$$
A_k(:,J)\,X = U_k\Sigma_k V_k(J,:)^{\top} W^{\top} = U_k\Sigma_k \bigl( W V_k(J,:) \bigr)^{\top} = U_k\Sigma_kV_k^{\top} = A_k ,
$$

because $WV_k(J,:) = V_k$.

*Step 4 — the error is carried by the tail.* Let $E = A - A_k$ and let $S_J$ be the
$n \times k$ column-selection matrix, so $M(:,J) = MS_J$. Using Step 3,

$$
A - A(:,J)X = A - \bigl( A_k(:,J) + E(:,J) \bigr) X = E - E S_J X = E\,(I_n - S_JX) .
$$

*Step 5 — bound.* $\lVert E \rVert_{\mathrm{op}} = \sigma_{k+1}$ by Theorem 4.6, and
$\lVert I_n - S_JX \rVert_{\mathrm{op}} \le 1 + \lVert X \rVert_{\mathrm{op}}$. Since
$\lVert X \rVert_{\mathrm{op}} \le \lVert X \rVert_F$ and $X$ has $k$ rows with at most $n$ entries
each bounded by $1$ — exactly $k$ of them forming $I_k$ and at most $k(n-k)$ others —

$$
\lVert X \rVert_F^2 \le k + k(n-k) = k(n-k+1) .
$$

$$
\boxed{\lVert A - A(:,J)X\rVert_{\mathrm{op}} \le \bigl( 1 + \sqrt{k(n-k+1)}\bigr)\sigma_{k+1}, \qquad X(:,J) = I_k, \ \lvert X_{ij}\rvert \le 1}
$$

**Key takeaway.** The interpolative decomposition writes $A$ in terms of $k$ of its own columns
with coefficients no larger than $1$, so sparsity and interpretability survive. Locating the exact
maximum-volume set is combinatorial; the practical substitute is a strong rank-revealing QR, which
achieves $\lVert A - A(:,J)X\rVert_{\mathrm{op}} \le \sqrt{1 + 4k(n-k)}\,\sigma_{k+1}$ with
$\lvert X_{ij}\rvert \le 2$ in polynomial time — Gu and Eisenstat, *SIAM Journal on Scientific
Computing* **17**(4), 1996, Theorem 3.2.

In [43]:
from itertools import combinations

m_i, n_i, k_i = 12, 8, 3
Ai = rng.standard_normal((m_i, 5)) @ rng.standard_normal((5, n_i))
Ui, si, Vti = np.linalg.svd(Ai, full_matrices=False)
Vk = Vti[:k_i].T

best_vol, Jbest = -1.0, None
for J in combinations(range(n_i), k_i):
    vol = abs(np.linalg.det(Vk[list(J), :]))
    if vol > best_vol:
        best_vol, Jbest = vol, list(J)

Xid = (Vk @ np.linalg.inv(Vk[Jbest, :])).T
err = np.linalg.norm(Ai - Ai[:, Jbest] @ Xid, 2)
bound = (1.0 + np.sqrt(k_i * (n_i - k_i + 1))) * si[k_i]
print("maximum volume            :", best_vol, "  columns J =", Jbest)
print("X(:,J) - I_k              : %.2e" % np.linalg.norm(Xid[:, Jbest] - np.eye(k_i)))
print("max |X_ij|                :", np.abs(Xid).max())
print("||A - A(:,J) X||_op       :", err)
print("sigma_{k+1}               :", si[k_i])
print("bound (1+sqrt(k(n-k+1))) s:", bound)
print("Gu-Eisenstat sqrt(1+4k(n-k)) s:", np.sqrt(1 + 4 * k_i * (n_i - k_i)) * si[k_i])
assert np.linalg.norm(Xid[:, Jbest] - np.eye(k_i)) < 1e-10
assert np.abs(Xid).max() <= 1.0 + 1e-9
assert err <= bound

maximum volume            : 0.3066756361845303   columns J = [0, 2, 4]
X(:,J) - I_k              : 1.74e-16
max |X_ij|                : 1.0
||A - A(:,J) X||_op       : 5.2184383966135535
sigma_{k+1}               : 3.8072426395644747
bound (1+sqrt(k(n-k+1))) s: 19.960005167916137
Gu-Eisenstat sqrt(1+4k(n-k)) s: 29.73551559175643
